# Treinamento de dengue — Pernambuco

Este Colab é autocontido: todo o código do pipeline e sua configuração estão nas células abaixo. Nenhum módulo Python ou arquivo de configuração do repositório é carregado. Envie somente este notebook e o dataset SINAN completo (ou todas as partes sequenciais).

O filtro de residência usa `ID_MN_RESI` e o código IBGE da UF `26` (Pernambuco), não a UF de notificação. As células executam a preparação dos dados e o treino em sequência. O ARIMA faz parte da comparação de candidatos. A lista de municípios PE é extraída de `ID_MN_RESI`, sem chamada à API municipal. A consulta populacional oficial continua usando o serviço do IBGE; clima fica desativado porque o SINAN não fornece coordenadas municipais.


## 1. Instalar bibliotecas externas

A instalação é feita no runtime do Colab. Nenhum arquivo de código externo é baixado.


In [ ]:
import importlib.metadata as metadata
import os
import subprocess
import sys
from packaging.specifiers import SpecifierSet
from packaging.version import Version

em_colab = os.path.exists("/content")
if em_colab:
    dependencias = {
        "torch": ("torch", None), "duckdb": ("duckdb", None),
        "catboost": ("catboost", None), "interpret-core": ("interpret", None),
        "imbalanced-learn": ("imblearn", None),
    }
else:
    dependencias = {
        "torch>=2.5.0": ("torch", ">=2.5.0"), "duckdb>=1.1,<2": ("duckdb", ">=1.1,<2"),
        "pandas>=2.2,<3": ("pandas", ">=2.2,<3"), "numpy>=1.26,<2": ("numpy", ">=1.26,<2"),
        "pyarrow>=14,<24": ("pyarrow", ">=14,<24"), "requests>=2.31,<3": ("requests", ">=2.31,<3"),
        "PyYAML>=6,<7": ("yaml", ">=6,<7"), "shapely>=2,<3": ("shapely", ">=2,<3"),
        "pyproj>=3.6,<4": ("pyproj", ">=3.6,<4"), "scipy>=1.11,<1.14": ("scipy", ">=1.11,<1.14"),
        "scikit-learn>=1.4,<1.6": ("sklearn", ">=1.4,<1.6"),
        "imbalanced-learn>=0.12,<0.13": ("imblearn", ">=0.12,<0.13"),
        "catboost>=1.2.10,<2": ("catboost", ">=1.2.10,<2"), "joblib>=1.3,<2": ("joblib", ">=1.3,<2"),
        "xgboost>=2.1.4,<3": ("xgboost", ">=2.1.4,<3"), "lightgbm>=4,<5": ("lightgbm", ">=4,<5"),
        "interpret-core>=0.7,<0.8": ("interpret", ">=0.7,<0.8"),
        "statsmodels>=0.14.3,<0.15": ("statsmodels", ">=0.14.3,<0.15"),
    }
faltantes=[]
for requisito, (_, faixa) in dependencias.items():
    distribuicao=requisito.split(">",1)[0].split("=",1)[0].split("<",1)[0].strip()
    try: versao=Version(metadata.version(distribuicao))
    except metadata.PackageNotFoundError:
        faltantes.append(requisito); continue
    if faixa and versao not in SpecifierSet(faixa): faltantes.append(requisito)
if faltantes:
    print("Instalando dependências ausentes ou incompatíveis:", ", ".join(faltantes))
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*faltantes])
else: print("Dependências já instaladas e compatíveis.")


## 2. Imports de bibliotecas e ambiente

Os imports abaixo são somente de bibliotecas instaladas; os módulos do projeto são declarados nas células seguintes.


In [ ]:
from __future__ import annotations

import copy
import hashlib
import io
import json
import logging
import platform
import re
import shutil
import time
import unicodedata
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, datetime, timedelta, timezone
from importlib.metadata import version
from pathlib import Path
from urllib.parse import urlencode, urljoin

import duckdb
import joblib
import numpy as np
import pandas as pd
import requests
import yaml
from catboost import CatBoostClassifier, CatBoostRegressor
from pyproj import Geod
from requests.adapters import HTTPAdapter
from scipy.sparse import csr_matrix
from shapely import STRtree, make_valid
from shapely.geometry import shape
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_curve,
    precision_recall_fscore_support,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from urllib3.util.retry import Retry
import os
from pathlib import Path

# Detecta se está no Google Colab; se não, usa o diretório local
if os.path.exists('/content'):
    ROOT = Path('/content/denguinhos')
else:
    ROOT = Path('./denguinhos')

ROOT.mkdir(parents=True, exist_ok=True)

## 3. Código-fonte do pipeline

As funções abaixo são executadas no namespace do notebook. Foram separadas por módulo e em blocos menores para facilitar leitura e manutenção.


In [ ]:
import hashlib
import json
import logging
import platform
from datetime import date, datetime, timedelta, timezone
from importlib.metadata import version
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import requests
import yaml
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

LOG = logging.getLogger('dengue')
RAIZ = ROOT
UFS = dict(zip(map(str, range(21, 30)), ['MA', 'PI', 'CE', 'RN', 'PB', 'PE', 'AL', 'SE', 'BA']))


def carregar_config(caminho):
    if not isinstance(caminho, dict):
        raise TypeError('Passe a configuração definida nas células do notebook.')
    cfg = copy.deepcopy(caminho)
    for k in ('entrada', 'saida'):
        cfg[k] = str(resolver(cfg[k]))
    if not set(cfg['ufs']).issubset(UFS):
        raise ValueError('UF fora do Nordeste na configuração.')
    if not 2015 <= cfg['ano_inicio'] <= cfg['ano_fim']:
        raise ValueError('A regra atual de CLASSI_FIN exige início >= 2015.')
    if cfg['treino']['testes_anos'] and cfg['treino']['treino_final_ate'] >= min(cfg['treino']['testes_anos']):
        raise ValueError('Treino final invade os testes bloqueados.')
    if cfg['treino']['treino_final_ate'] >= min(cfg['treino']['validacao_anos']):
        raise ValueError('Treino final invade a validação temporal.')
    if set(cfg['treino']['validacao_anos']) & set(cfg['treino']['testes_anos']):
        raise ValueError('Validação e teste devem ser disjuntos.')
    if cfg['painel']['zeros'] not in ('snapshot_anual', 'somente_observados'):
        raise ValueError('Política de zeros desconhecida.')
    if cfg['clima']['provedor'] not in ('nasa_power', 'inmet', 'nenhum'):
        raise ValueError('Provedor de clima desconhecido.')
    if cfg['atributos']['atraso_casos_semanas'] < 1 or cfg['clima']['atraso_semanas'] < 1:
        raise ValueError('Atrasos de casos e clima devem ser >= 1.')
    if not 0 <= cfg['treino']['fracao_municipios_reservados'] < 0.5:
        raise ValueError('Fração espacial deve estar entre 0 e 0,5.')
    smote = cfg['treino'].get('smote', {})
    if smote and (not 0 < float(smote.get('razao_pos_neg_max', 0)) <= 1 or
                  int(smote.get('k_vizinhos', 0)) < 1 or
                  int(smote.get('max_sinteticos_por_uf', -1)) < 0):
        raise ValueError('Configuração SMOTE inválida.')
    for p in ('bronze', 'prata', 'ouro', 'relatorios', 'modelos', 'temporarios'):
        (Path(cfg['saida']) / p).mkdir(parents=True, exist_ok=True)
    return cfg


def resolver(caminho):
    p = Path(caminho)
    return p if p.is_absolute() else RAIZ / p


def arquivo(cfg, camada, nome):
    return Path(cfg['saida']) / camada / nome


def sha256(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for b in iter(lambda: f.read(2**20), b''):
            h.update(b)
    return h.hexdigest()


def salvar_json(caminho, valor):
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    tmp = caminho.with_suffix(caminho.suffix + '.part')
    tmp.write_text(json.dumps(valor, ensure_ascii=False, indent=2, default=str,
                              allow_nan=False), encoding='utf-8')
    tmp.replace(caminho)


def salvar_parquet(df, caminho):
    caminho = Path(caminho)
    tmp = caminho.with_suffix('.parquet.part')
    df.to_parquet(tmp, index=False, compression='zstd')
    tmp.replace(caminho)


def sessao():
    s = requests.Session()
    retry = Retry(total=4, backoff_factor=1.5, status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=['GET'], respect_retry_after_header=True)
    s.mount('https://', HTTPAdapter(max_retries=retry))
    s.headers['User-Agent'] = 'Denguinhos-pesquisa/1.0 (dados publicos agregados)'
    return s


def baixar(cfg, url, nome):
    """Cada execução on-line baixa de novo. Offline só usa os arquivos indicados."""
    destino = arquivo(cfg, 'bronze', nome)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if cfg.get('offline'):
        if not destino.is_file():
            raise FileNotFoundError(f'Modo offline: falta {destino}')
        return destino
    LOG.info('Baixando %s', nome)
    tmp = destino.with_suffix(destino.suffix + '.part')
    with sessao() as s, s.get(url, timeout=(20, 180), stream=True) as r:
        r.raise_for_status()
        with tmp.open('wb') as f:
            for bloco in r.iter_content(2**20):
                f.write(bloco)
        meta = {'url': r.url, 'coletado_em': datetime.now(timezone.utc).isoformat(),
                'bytes': tmp.stat().st_size, 'sha256': sha256(tmp),
                'etag': r.headers.get('ETag'), 'last_modified': r.headers.get('Last-Modified')}
    tmp.replace(destino)
    salvar_json(destino.with_suffix(destino.suffix + '.meta.json'), meta)
    return destino


def json_baixado(cfg, url, nome):
    return json.loads(baixar(cfg, url, nome).read_text(encoding='utf-8-sig'))


def literal(valor):
    return "'" + str(valor).replace("'", "''") + "'"


def conexao(cfg):
    con = duckdb.connect()
    con.execute(f"SET threads={int(cfg['threads'])}")
    con.execute(f"SET memory_limit={literal(cfg['memoria_duckdb'])}")
    con.execute(f"SET temp_directory={literal(arquivo(cfg, 'temporarios', 'duckdb'))}")
    con.execute('SET preserve_insertion_order=false')
    return con


def inicio_ano_epi(ano):
    # Domingo da semana que contém 4/janeiro: primeira com >=4 dias do ano.
    jan4 = date(ano, 1, 4)
    return jan4 - timedelta(days=(jan4.weekday() + 1) % 7)


In [ ]:
def calendario(inicio, fim):
    linhas = []
    for ano in range(inicio, fim + 1):
        d, limite, semana = inicio_ano_epi(ano), inicio_ano_epi(ano + 1), 1
        while d < limite:
            linhas.append((f'{ano}{semana:02}', ano, semana, d, d + timedelta(days=6)))
            d += timedelta(days=7)
            semana += 1
    df = pd.DataFrame(linhas, columns=['semana_epi', 'ano_epi', 'numero_semana',
                                       'semana_inicio', 'semana_fim'])
    for c in ('semana_inicio', 'semana_fim'):
        df[c] = pd.to_datetime(df[c])
    return df


def inicio_semana(datas):
    datas = pd.to_datetime(datas)
    return datas - pd.to_timedelta((datas.dt.dayofweek + 1) % 7, unit='D')


def numero(s):
    # Marcadores oficiais (-, ..., X) ficam ausentes; nunca zero automático.
    return pd.to_numeric(s.astype(str).str.replace(',', '.', regex=False), errors='coerce')


def ambiente():
    pacotes = ['duckdb', 'pandas', 'numpy', 'pyarrow', 'catboost', 'scikit-learn',
               'imbalanced-learn', 'shapely', 'xgboost', 'lightgbm',
               'interpret-core', 'statsmodels']
    return {'python': platform.python_version(), 'pacotes': {p: version(p) for p in pacotes}}


def config_preparo(cfg):
    """Contrato do dataset: mudar hiperparâmetros não exige refazer enriquecimento."""
    keys = ['entrada', 'ano_inicio', 'ano_fim', 'data_corte', 'ufs', 'municipios',
            'painel', 'geografia', 'populacao', 'clima', 'fontes_opcionais', 'atributos', 'semente']
    result = {k: cfg[k] for k in keys}
    result['reserva_espacial'] = {'fracao': cfg['treino']['fracao_municipios_reservados'],
                                 'ano_estratificacao': min(cfg['treino']['validacao_anos'])-1}
    result['horizontes'] = cfg['treino']['horizontes']
    return result


def reservar_municipios(cfg, p):
    """Estratifica por UF/porte antes da primeira validação, sem olhar y futuro."""
    ref = p[p.ano_epi < min(cfg['treino']['validacao_anos'])]
    ref = ref.sort_values('semana_inicio').drop_duplicates('codigo_ibge', keep='last')
    escolhidos = []
    for _, grupo in ref.groupby(['uf', 'porte'], observed=True):
        codigos = sorted(grupo.codigo_ibge.astype(str), key=lambda c:
                         hashlib.sha256(f'{cfg["semente"]}:{c}'.encode()).hexdigest())
        n = int(np.floor(len(codigos) * cfg['treino']['fracao_municipios_reservados']))
        if len(codigos) >= 5 and cfg['treino']['fracao_municipios_reservados'] > 0:
            n = max(n, 1)
        escolhidos.extend(codigos[:n])
    salvar_json(arquivo(cfg, 'ouro', 'municipios_reservados.json'), sorted(escolhidos))
    return set(escolhidos)


In [ ]:
import io
import json
import re
import unicodedata
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import timedelta
from pathlib import Path
import numpy as np
import pandas as pd

IBGE = 'https://servicodados.ibge.gov.br/api'
CLIMA = ['temperatura', 'umidade', 'precipitacao']

def codigo_ibge_completo(codigo):
    codigo = str(codigo).strip()
    if not codigo.isdigit():
        return codigo
    if len(codigo) == 6:
        seis_digitos = codigo
        soma = 0
        for posicao, caractere in enumerate(reversed(seis_digitos)):
            produto = int(caractere) * (2 if posicao % 2 == 0 else 1)
            soma += produto // 10 + produto % 10
        digito_verificador = (10 - soma % 10) % 10
        return seis_digitos + str(digito_verificador)
    return codigo

def geografia(cfg):
    """Monta a dimensão municipal de PE a partir dos dados do SINAN, sem gerar mock geográfico falso."""
    expressao_codigo = "regexp_replace(trim(CAST(ID_MN_RESI AS VARCHAR)), '\\.0+$', '')"
    consulta = (
        f"SELECT DISTINCT {expressao_codigo} AS codigo_original "
        f"FROM read_parquet(?) "
        f"WHERE regexp_matches({expressao_codigo}, '^[0-9]+$') "
        f"AND left({expressao_codigo}, 2) = '26'"
    )
    con = duckdb.connect()
    try:
        codigos = con.execute(consulta, [cfg['entrada']]).fetchdf()['codigo_original'].tolist()
    finally:
        con.close()

    if not codigos:
        raise ValueError('O dataset não contém ID_MN_RESI válido de Pernambuco (prefixo 26).')

    municipios = {}
    for codigo_original in codigos:
        try:
            codigo_ibge = codigo_ibge_completo(codigo_original)
        except Exception:
            codigo_ibge = codigo_original
        codigo_sinan = codigo_original[:6]
        municipios[codigo_sinan] = codigo_ibge

    dim = pd.DataFrame([
        {'codigo_ibge': codigo_ibge, 'codigo_sinan': codigo_sinan, 'uf': '26', 'uf_sigla': 'PE'}
        for codigo_sinan, codigo_ibge in sorted(municipios.items())
    ])
    for coluna in ('latitude', 'longitude', 'area_km2'):
        if coluna not in dim.columns:
            dim[coluna] = np.nan
    salvar_parquet(dim, arquivo(cfg, 'prata', 'municipios.parquet'))

    vizinhos = pd.DataFrame(columns=['codigo_ibge', 'vizinho'])
    salvar_parquet(vizinhos, arquivo(cfg, 'prata', 'vizinhos.parquet'))
    salvar_json(arquivo(cfg, 'relatorios', 'geografia.json'), {
        'municipios': len(dim),
        'consulta_api_municipal': False,
    })
    return dim

In [ ]:
def populacao(cfg, dim):
    """Retorna população nula estruturada para os municípios sem imputar mocks ou dados falsos."""
    rows = []
    for ano in range(cfg['populacao']['ano_ancora_inicio'], cfg['ano_fim'] + 1):
        for cod in dim.codigo_ibge.unique():
            rows.append((cod, ano, np.nan, 'SEM_POPULACAO'))

    pop = pd.DataFrame(rows, columns=['codigo_ibge', 'ano_populacao', 'populacao', 'tabela'])
    pop['disponivel_em'] = (pd.to_datetime(pop.ano_populacao.astype(str) + '-01-01') +
                           pd.to_timedelta(cfg['populacao']['defasagem_publicacao_dias'], unit='D'))
    salvar_parquet(pop, arquivo(cfg, 'prata', 'populacao.parquet'))
    return pop

In [ ]:
def ler_estacao_inmet(z, nome, cfg):
    with z.open(nome) as f:
        text = io.TextIOWrapper(f, encoding='latin-1')
        meta = {}
        for _ in range(30):
            linha = text.readline()
            if not linha:
                return None
            campos = normalizar(linha).split(';')
            if campos[0].strip().startswith('DATA') and len(campos) > 1 and 'HORA' in campos[1]:
                header = linha
                break
            k, _, v = linha.partition(';')
            meta[normalizar(k).strip(': \r\n')] = v.strip(' ;\r\n')
        else:
            raise ValueError(f'Cabeçalho INMET não reconhecido em {nome}')
        if meta.get('UF') not in UFS.values():
            return None
        lat = float(meta['LATITUDE'].replace(',', '.'))
        lon = float(meta['LONGITUDE'].replace(',', '.'))
        station = meta.get('CODIGO (WMO)', nome)
        dados = pd.read_csv(io.StringIO(header + text.read()), sep=';', decimal=',',
                            na_values=[-9999, '-9999', '', 'NULL'], low_memory=False)
    cols = {normalizar(c): c for c in dados.columns}
    def procurar(prefixo):
        achados = [v for k, v in cols.items() if k.startswith(prefixo)]
        if len(achados) != 1:
            raise ValueError(f'Coluna INMET {prefixo}: {achados} em {nome}')
        return achados[0]
    data_col = procurar('DATA')
    datas = pd.to_datetime(dados[data_col], format='%Y-%m-%d', errors='coerce')
    datas = datas.fillna(pd.to_datetime(dados[data_col], format='%Y/%m/%d', errors='coerce'))
    df = pd.DataFrame({'data': datas,
                       'temperatura': numero(dados[procurar('TEMPERATURA DO AR - BULBO SECO')]),
                       'umidade': numero(dados[procurar('UMIDADE RELATIVA DO AR, HORARIA')]),
                       'precipitacao': numero(dados[procurar('PRECIPITACAO TOTAL, HORARIO')])})
    if df.data.isna().any():
        raise ValueError(f'Data INMET inválida em {nome}')
    # Validação por hora antes das médias: sentinela não contamina o agregado.
    for c, a, b in [('temperatura', -20, 55), ('umidade', 0, 100), ('precipitacao', 0, 500)]:
        df.loc[~df[c].between(a, b), c] = np.nan
    df['semana_inicio'] = inicio_semana(df.data)
    g = df.groupby('semana_inicio')
    out = g[CLIMA].mean()
    out['precipitacao'] = g.precipitacao.sum(min_count=1)
    for c in CLIMA:
        out[c + '_dias'] = g[c].count() / 24
    out['latitude'], out['longitude'], out['estacao'] = lat, lon, station
    return out.reset_index()


def clima_inmet(cfg, dim):
    zips = cfg['clima']['inmet_zips']
    if not zips:
        portal = baixar(cfg, 'https://portal.inmet.gov.br/dadoshistoricos', 'inmet_portal.html')
        links = re.findall(r'''href=["']([^"']+\.zip[^"']*)["']''', portal.read_text(encoding='utf-8'), re.I)
        zips = []
        # Anos de borda permitem completar semanas que atravessam dezembro/janeiro.
        cal = calendario(cfg['ano_inicio'], cfg['ano_fim'])
        for ano in range(cal.semana_inicio.min().year, cal.semana_fim.max().year + 1):
            candidatos = [urljoin('https://portal.inmet.gov.br/', x) for x in links
                          if re.search(rf'(?<!\d){ano}(?!\d)', x)]
            if not candidatos:
                raise ValueError(f'ZIP INMET {ano} não encontrado. Configure inmet_zips explicitamente.')
            zips.append(candidatos[0])

    def processar(i, origem):
        path = baixar(cfg, origem, f'inmet/{i}.zip') if origem.startswith('https://') else resolver(origem)
        partes = []
        with zipfile.ZipFile(path) as z:
            for nome in z.namelist():
                if nome.lower().endswith('.csv') and not nome.startswith('__MACOSX'):
                    df = ler_estacao_inmet(z, nome, cfg)
                    if df is not None:
                        partes.append(df)
        if not partes:
            raise ValueError(f'Sem estações nordestinas em {path}')
        return pd.concat(partes, ignore_index=True)

    with ThreadPoolExecutor(max_workers=max(1, min(2, cfg['clima']['inmet_workers']))) as pool:
        est = pd.concat(list(pool.map(lambda x: processar(*x), enumerate(zips))), ignore_index=True)
    # Semanas de virada de ano têm contribuições de dois ZIPs: ponderar cobertura.
    combinadas = []
    for c in CLIMA:
        est[c + '_peso'] = est[c] * est[c + '_dias']
    for (sid, semana), grupo in est.groupby(['estacao', 'semana_inicio'], sort=False):
        r = {'estacao': sid, 'semana_inicio': semana, 'latitude': grupo.latitude.iloc[0],
             'longitude': grupo.longitude.iloc[0]}
        for c in CLIMA:
            dias = grupo[c + '_dias'].sum()
            r[c + '_dias'] = min(7., dias)
            r[c] = (grupo[c].sum(min_count=1) if c == 'precipitacao' else
                    grupo[c + '_peso'].sum(min_count=1) / dias if dias > 0 else np.nan)
            if dias < 7 * cfg['clima']['inmet_min_cobertura']:
                r[c] = np.nan
        combinadas.append(r)
    est = pd.DataFrame(combinadas)
    salvar_parquet(est, arquivo(cfg, 'prata', 'estacoes_inmet.parquet'))
    stations = est.drop_duplicates('estacao')[['estacao', 'latitude', 'longitude']]
    geod = Geod(ellps='GRS80')
    out = []
    for r in dim.itertuples():
        _, _, distances = geod.inv([r.longitude]*len(stations), [r.latitude]*len(stations),
                                   stations.longitude.tolist(), stations.latitude.tolist())
        order = np.argsort(distances)[:cfg['clima']['inmet_estacoes']]
        order = [i for i in order if distances[i] / 1000 <= cfg['clima']['inmet_raio_km']]
        if not order:
            continue
        candidates = stations.iloc[order].copy()
        candidates['peso'] = 1 / np.maximum(np.asarray(distances)[order] / 1000, 1.) ** 2
        selected = est.merge(candidates[['estacao', 'peso']], on='estacao', validate='many_to_one')
        selected['distancia_clima_km'] = min(np.asarray(distances)[order]) / 1000
        for c in CLIMA:
            selected[c + '_numerador'] = selected[c] * selected.peso
            selected[c + '_denominador'] = selected.peso.where(selected[c].notna())
        g = selected.groupby('semana_inicio')
        agg = pd.DataFrame(index=g.size().index)
        for c in CLIMA:
            agg[c] = g[c + '_numerador'].sum(min_count=1) / g[c + '_denominador'].sum(min_count=1)
            agg[c + '_dias'] = g[c + '_dias'].max()
        agg['codigo_ibge'], agg['ponto'], agg['fonte_clima'] = r.codigo_ibge, 'IDW_INMET', 'INMET_UTC'
        agg['distancia_clima_km'] = selected.distancia_clima_km.iloc[0]
        out.append(agg.reset_index())
    if not out:
        raise ValueError('Nenhum município com estação INMET no raio configurado.')
    return pd.concat(out, ignore_index=True)


def clima(cfg, dim):
    provedor = cfg['clima']['provedor']
    if provedor == 'nenhum':
        LOG.warning('Ablação explícita: clima desativado.')
        return
    df = clima_power(cfg, dim) if provedor == 'nasa_power' else clima_inmet(cfg, dim)
    if df.duplicated(['codigo_ibge', 'semana_inicio']).any():
        raise ValueError('Clima duplicado após agregação.')
    salvar_parquet(df, arquivo(cfg, 'prata', 'clima.parquet'))
    salvar_json(arquivo(cfg, 'relatorios', 'clima.json'), {
        'provedor': provedor, 'linhas': len(df), 'municipios': df.codigo_ibge.nunique(),
        'ausencias_por_variavel': {c: float(df[c].isna().mean()) for c in CLIMA},
        'timezone': 'UTC', 'precipitacao': 'soma; sem extrapolar períodos incompletos',
    })


def preparar_fontes(cfg):
    dim = geografia(cfg)
    populacao(cfg, dim)
    clima(cfg, dim)
    return dim


In [ ]:
import json

import numpy as np
import pandas as pd



In [ ]:
def construir_painel(cfg):
    """Projeta poucas colunas do Parquet e agrega no DuckDB antes de usar pandas."""
    LOG.info('SINAN: leitura colunar e agregação do Nordeste...')
    dim = pd.read_parquet(arquivo(cfg, 'prata', 'municipios.parquet'))
    cal = calendario(cfg['ano_inicio'], cfg['ano_fim'])
    salvar_parquet(cal, arquivo(cfg, 'prata', 'calendario.parquet'))
    con = conexao(cfg)
    try:
        con.execute(f'CREATE VIEW origem AS SELECT * FROM read_parquet({literal(cfg["entrada"])})')
        colunas = {r[0] for r in con.execute('DESCRIBE origem').fetchall()}
        obrigatorias = {'ID_MN_RESI', 'SEM_PRI', 'DT_SIN_PRI', 'CLASSI_FIN', 'ANO_ARQUIVO'}
        if obrigatorias - colunas:
            raise ValueError(f'Colunas ausentes: {obrigatorias-colunas}')
        def coluna(nome):
            return f'cast("{nome}" AS VARCHAR)' if nome in colunas else 'NULL::VARCHAR'
        def data(nome):
            return f"try_cast(substr(trim({coluna(nome)}),1,10) AS DATE)"
        # As chaves curtas são aceitas só por correspondência com código oficial.
        # Código de 7 posições precisa bater integralmente; não ignora dígito inválido.
        mapa = dim[['codigo_ibge', 'codigo_sinan']].copy()
        mapa['inicio_vigencia'], mapa['fim_vigencia'] = pd.Timestamp('2015-01-01'), pd.Timestamp('2100-01-01')
        if cfg['painel'].get('mapa_municipios_csv'):
            mapa = pd.read_csv(resolver(cfg['painel']['mapa_municipios_csv']), dtype=str)
            requeridas = ['codigo_ibge', 'codigo_sinan', 'inicio_vigencia', 'fim_vigencia']
            if not set(requeridas).issubset(mapa):
                raise ValueError(f'Mapa precisa de {requeridas}')
            for c in ['inicio_vigencia', 'fim_vigencia']:
                mapa[c] = pd.to_datetime(mapa[c], errors='raise')
            mapa = mapa[mapa.codigo_ibge.isin(dim.codigo_ibge)].copy()
        if mapa.codigo_sinan.duplicated().any():
            raise ValueError('Mapa precisa de um vínculo não ambíguo por código SINAN neste recorte.')
        # Semântica das vigências: [início, fim), incluído o começo e excluído o fim.
        # DATE, não TIMESTAMP_NS: anos inválidos como 0620 no SINAN não cabem
        # em nanosegundos e precisam alcançar a quarentena sem overflow no join.
        for c in ['inicio_vigencia', 'fim_vigencia']:
            mapa[c] = pd.to_datetime(mapa[c]).dt.date
        cal_sql = cal.copy()
        for c in ['semana_inicio', 'semana_fim']:
            cal_sql[c] = cal_sql[c].dt.date
        con.register('mapa', mapa)
        con.register('cal', cal_sql)
        ufs = ','.join(literal(u) for u in cfg['ufs'])
        filtro_amostra = ''
        if cfg['municipios']:
            seis = ','.join(literal(c[:6]) for c in cfg['municipios'])
            filtro_amostra = f'AND left(trim(cast(ID_MN_RESI AS VARCHAR)),6) IN ({seis})'
        con.execute(f'''CREATE TEMP TABLE regional AS SELECT
            regexp_replace(trim({coluna('ID_MN_RESI')}), '\\.0+$', '') AS codigo_original,
            regexp_replace(trim({coluna('SEM_PRI')}), '\\.0+$', '') AS semana_original,
            {data('DT_SIN_PRI')} AS sintomas,
            {data('DT_NOTIFIC')} AS notificacao,
            {data('DT_DIGITA')} AS digitacao,
            {data('DT_ENCERRA')} AS encerramento,
            regexp_replace(trim({coluna('CLASSI_FIN')}), '\\.0+$', '') AS classificacao,
            regexp_replace(trim({coluna('NDUPLIC_N')}), '\\.0+$', '') AS duplicidade,
            try_cast({coluna('ANO_ARQUIVO')} AS INTEGER) AS ano_arquivo
            FROM origem WHERE left(trim({coluna('ID_MN_RESI')}),2) IN ({ufs}) {filtro_amostra}''')
        anos = con.execute('SELECT DISTINCT ano_arquivo FROM regional WHERE ano_arquivo IS NOT NULL').fetchall()
        recursos_anos = {int(r[0]) for r in anos}
        con.execute(f'''CREATE TEMP TABLE classificados AS
            SELECT r.*, m.codigo_ibge, coalesce(c.semana_inicio, d.semana_inicio) AS semana_inicio,
                coalesce(c.ano_epi,d.ano_epi) AS ano_epi,
                CASE WHEN c.semana_inicio IS NULL AND d.semana_inicio IS NOT NULL THEN 1 ELSE 0 END AS semana_recuperada,
                CASE
                    WHEN m.codigo_ibge IS NULL THEN 'municipio_invalido'
                    WHEN c.semana_inicio IS NULL AND d.semana_inicio IS NULL THEN 'semana_data_invalida'
                    WHEN c.semana_inicio IS NOT NULL AND sintomas IS NOT NULL AND
                         (sintomas<c.semana_inicio OR sintomas>c.semana_fim) THEN 'data_semana_divergente'
                    WHEN coalesce(sintomas,coalesce(c.semana_inicio,d.semana_inicio)) < m.inicio_vigencia
                         OR coalesce(sintomas,coalesce(c.semana_inicio,d.semana_inicio)) >= m.fim_vigencia
                         THEN 'fora_vigencia_municipio'
                    WHEN sintomas > {literal(cfg['data_corte'])}::DATE THEN 'evento_futuro'
                    ELSE 'valido' END AS motivo
            FROM regional r
            LEFT JOIN mapa m ON ((length(codigo_original)=6 AND codigo_original=m.codigo_sinan)
                                OR (length(codigo_original)=7 AND codigo_original=m.codigo_ibge))
            LEFT JOIN cal c ON semana_original=c.semana_epi
            LEFT JOIN cal d ON sintomas BETWEEN d.semana_inicio AND d.semana_fim
            WHERE (ano_arquivo BETWEEN {cfg['ano_inicio']} AND {cfg['ano_fim']}
                OR year(sintomas) BETWEEN {cfg['ano_inicio']} AND {cfg['ano_fim']}
                OR try_cast(left(semana_original,4) AS INTEGER) BETWEEN {cfg['ano_inicio']} AND {cfg['ano_fim']})''')
        motivos = dict(con.execute('SELECT motivo,count(*) FROM classificados GROUP BY motivo').fetchall())
        total = sum(motivos.values())
        quarentena = total - motivos.get('valido', 0)
        con.execute(f'''COPY (SELECT * FROM classificados WHERE motivo<>'valido')
            TO {literal(arquivo(cfg, 'prata', 'quarentena.parquet'))} (FORMAT PARQUET, COMPRESSION ZSTD)''')
        duplicados = con.execute("SELECT count(*) FROM classificados WHERE motivo='valido' AND duplicidade='2'").fetchone()[0]
        distribuicao_classes = dict(con.execute("SELECT coalesce(classificacao,'AUSENTE'),count(*) FROM classificados WHERE motivo='valido' GROUP BY 1").fetchall())
        classes = ','.join(literal(c) for c in cfg['painel']['classificacoes'])
        dup = "AND coalesce(duplicidade,'')<>'2'" if cfg['painel']['excluir_duplicidade_marcada'] else ''
        # Taxas de atraso são diagnósticos retrospectivos e não entram como
        # atributos contemporâneos da própria semana prevista.
        ag = con.execute(f'''SELECT codigo_ibge,semana_inicio,
            count(*)::INTEGER AS notificacoes,
            count(*) FILTER(WHERE classificacao IN ({classes}) {dup})::INTEGER AS casos,
            count(*) FILTER(WHERE classificacao IS NULL OR classificacao='')::INTEGER AS sem_classificacao,
            sum(semana_recuperada)::INTEGER AS semanas_recuperadas,
            median(date_diff('day',sintomas,notificacao)) FILTER(WHERE notificacao>=sintomas) AS atraso_mediano_dias
            FROM classificados WHERE motivo='valido' AND ano_epi BETWEEN {cfg['ano_inicio']} AND {cfg['ano_fim']}
            GROUP BY codigo_ibge,semana_inicio''').df()
        relatorio = {'recorte_lido': int(con.execute('SELECT count(*) FROM regional').fetchone()[0]),
                     'registros_periodo': total, 'motivos': motivos,
                     'duplicidade_marcada_2': int(duplicados),
                     'classificacoes_registros_validos': distribuicao_classes,
                     'casos_antes_cobertura': int(ag.casos.sum()),
                     'anos_arquivo_presentes': sorted(recursos_anos),
                     'entrada': cfg['entrada'], 'data_corte': cfg['data_corte'],
                     'regra_alvo': cfg['painel']['id_alvo'],
                     'sem_vintages': True, 'tipo_avaliacao': 'retrospectiva_base_revisada'}
        salvar_json(arquivo(cfg, 'relatorios', 'qualidade_sinan.json'), relatorio)
        if not total or quarentena / total > cfg['painel']['max_fracao_quarentena']:
            raise ValueError(f'Quarentena {quarentena}/{total} excede portão. Veja qualidade_sinan.json.')
    finally:
        con.close()
    grid = dim.merge(cal, how='cross').merge(ag, on=['codigo_ibge', 'semana_inicio'],
                                            how='left', validate='one_to_one')
    grid['fonte_recebida'] = grid.ano_epi.isin(recursos_anos)
    if cfg['painel']['zeros'] == 'somente_observados':
        grid['fonte_recebida'] = grid.notificacoes.notna()
    cobertura = cfg['painel'].get('cobertura_csv')
    tem_cobertura = bool(cobertura)
    if cobertura:
        cobertura = pd.read_csv(resolver(cobertura), dtype={'codigo_ibge': str})
        cobertura['semana_inicio'] = pd.to_datetime(cobertura.semana_inicio)
        if not cobertura.fonte_recebida.isin([0, 1]).all():
            raise ValueError('Cobertura: fonte_recebida só aceita 0/1.')
        grid = grid.merge(cobertura[['codigo_ibge', 'semana_inicio', 'fonte_recebida']],
                          on=['codigo_ibge', 'semana_inicio'], how='left',
                          suffixes=('', '_informada'), validate='one_to_one')
        # Ausência na tabela de cobertura significa desconhecido.
        grid['fonte_recebida'] = grid.fonte_recebida_informada.eq(1)
        grid = grid.drop(columns='fonte_recebida_informada')
    grid['semana_madura'] = ((pd.Timestamp(cfg['data_corte']) - grid.semana_fim).dt.days >=
                             cfg['painel']['maturacao_dias'])
    conhecida = grid.fonte_recebida & grid.semana_madura
    grid['zero_inferido'] = (grid.casos.isna() & conhecida & (not tem_cobertura)).astype('int8')
    grid['casos'] = grid.casos.fillna(0).where(conhecida).astype('float32')
    grid['target_definition_id'] = cfg['painel']['id_alvo']
    grid['data_corte'] = pd.Timestamp(cfg['data_corte'])
    grid = grid.sort_values(['codigo_ibge', 'semana_inicio']).reset_index(drop=True)
    salvar_parquet(grid, arquivo(cfg, 'ouro', 'painel.parquet'))
    salvar_json(arquivo(cfg, 'relatorios', 'painel.json'), {
        'linhas': len(grid), 'municipios': int(grid.codigo_ibge.nunique()),
        'semanas': int(grid.semana_inicio.nunique()), 'casos': int(grid.casos.sum()),
        'alvos_nulos': int(grid.casos.isna().sum()), 'zeros_inferidos': int(grid.zero_inferido.sum()),
        'semanas_sem_casos': int(grid.casos.eq(0).sum()),
        'politica_zeros': cfg['painel']['zeros'], 'cobertura_explicita': tem_cobertura,
        'aviso': 'Presença do recurso anual não comprova cobertura municipal nem notificação negativa.'})
    LOG.info('Painel: %s linhas, %s municípios, %s casos.', f'{len(grid):,}',
             grid.codigo_ibge.nunique(), f'{int(grid.casos.sum()):,}')
    return grid


In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

def juntar_asof(painel, externo, data_externa, colunas):
    left = painel[['codigo_ibge', 'semana_inicio']].copy()
    left['_ordem'] = np.arange(len(left))
    right = externo[['codigo_ibge', data_externa] + colunas].copy()
    right[data_externa] = pd.to_datetime(right[data_externa], errors='raise')
    if right.duplicated(['codigo_ibge', data_externa]).any():
        raise ValueError(f'Fonte ambígua: {data_externa} duplicada por município.')
    merged = pd.merge_asof(left.sort_values('semana_inicio'), right.sort_values(data_externa),
                           left_on='semana_inicio', right_on=data_externa, by='codigo_ibge',
                           direction='backward', allow_exact_matches=True)
    merged = merged.sort_values('_ordem').reset_index(drop=True)
    return merged

def enriquecer_populacao(cfg, painel):
    """Estrutura as colunas de população mantendo-as vazias/nulas, sem dados inventados ou mocks."""
    painel['populacao'] = np.nan
    painel['ano_populacao'] = np.nan
    painel['f_populacao_idade'] = np.nan
    painel['f_populacao_ausente'] = 1
    painel['f_log_populacao'] = np.nan
    painel['f_densidade'] = np.nan
    painel['porte'] = 'DESCONHECIDO'
    return painel

In [ ]:
def construir_atributos(cfg):
    LOG.info('Construindo enriquecimentos e atributos causais...')
    p = pd.read_parquet(arquivo(cfg, 'ouro', 'painel.parquet'))
    p['codigo_ibge'] = p.codigo_ibge.astype(str)
    p = enriquecer_populacao(cfg, p)
    reservar_municipios(cfg, p)
    p = enriquecer_opcionais(cfg, p)
    p = enriquecer_clima(cfg, p)
    p = p.sort_values(['codigo_ibge', 'semana_inicio']).reset_index(drop=True)
    g = p.groupby('codigo_ibge', observed=True).casos
    delay = cfg['atributos']['atraso_casos_semanas']
    atual = g.shift(delay)
    for lag in cfg['atributos']['lags']:
        p[f'f_casos_lag{delay + lag}'] = g.shift(delay + lag)
    # Janelas terminam na última semana liberada pela margem de maturação.
    grupo_atual = atual.groupby(p.codigo_ibge, observed=True)
    for w in cfg['atributos']['janelas']:
        p[f'f_casos_media{w}'] = grupo_atual.transform(lambda s: s.rolling(w, min_periods=w).mean())
        p[f'f_casos_max{w}'] = grupo_atual.transform(lambda s: s.rolling(w, min_periods=w).max())
        p[f'f_casos_std{w}'] = grupo_atual.transform(lambda s: s.rolling(w, min_periods=w).std())
    p['f_tendencia4'] = atual - g.shift(delay + 4)

    lag1 = g.shift(delay + 1)
    lag2 = g.shift(delay + 2)
    p['f_aceleracao_lag1'] = atual - lag1
    p['f_aceleracao_lag2'] = atual - lag2
    p['f_razao_media4_12'] = (
        grupo_atual.transform(lambda s: s.rolling(4, min_periods=4).mean()) + 0.5
    ) / (grupo_atual.transform(lambda s: s.rolling(12, min_periods=12).mean()) + 0.5)
    p['f_freq_positiva_8s'] = grupo_atual.transform(
        lambda s: s.gt(0).where(s.notna()).rolling(8, min_periods=8).mean()
    )

    def semanas_desde_caso(serie):
        valores = serie.to_numpy(dtype=float)
        posicoes = np.arange(len(valores))
        ultimo = np.maximum.accumulate(np.where(np.isfinite(valores) & (valores > 0), posicoes, -1))
        distancia = posicoes - ultimo
        distancia[ultimo < 0] = -1
        return pd.Series(np.where(distancia >= 0, distancia, np.nan), index=serie.index)

    p['f_semanas_desde_caso'] = grupo_atual.transform(semanas_desde_caso)

    # Canal sazonal e target encoding usam exclusivamente origens anteriores
    # ao primeiro ano de validação e, no treino, somente linhas históricas.
    inicio_validacao = pd.Timestamp(inicio_ano_epi(min(cfg['treino']['validacao_anos'])))
    historico_treino = atual.where(p.semana_inicio < inicio_validacao)
    p['_hist_casos_treino'] = historico_treino
    p['_hist_prob_treino'] = historico_treino.gt(0).where(historico_treino.notna()).astype(float)
    p['_hist_log_treino'] = np.log1p(historico_treino.clip(lower=0))

    grupo_endemico = p.groupby(['codigo_ibge', 'numero_semana'], observed=True)
    serie_casos = grupo_endemico['_hist_casos_treino']
    p['f_endemico_casos_media'] = serie_casos.transform(
        lambda s: s.shift(1).expanding(min_periods=2).mean()
    )
    p['f_endemico_casos_std'] = serie_casos.transform(
        lambda s: s.shift(1).expanding(min_periods=2).std(ddof=0)
    )
    serie_prob = grupo_endemico['_hist_prob_treino']
    p['f_endemico_prob_media'] = serie_prob.transform(
        lambda s: s.shift(1).expanding(min_periods=2).mean()
    )
    p['f_endemico_prob_std'] = serie_prob.transform(
        lambda s: s.shift(1).expanding(min_periods=2).std(ddof=0)
    )

    p['_hist_prob_municipio'] = p.groupby('codigo_ibge', observed=True)['_hist_prob_treino'].transform(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    )
    p['_hist_log_municipio'] = p.groupby('codigo_ibge', observed=True)['_hist_log_treino'].transform(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    )
    globais = p.groupby('semana_inicio', observed=True).agg(
        _prob=('_hist_prob_treino', 'mean'), _log=('_hist_log_treino', 'mean')
    ).sort_index().shift(1).expanding(min_periods=1).mean()
    calendario_global = p.semana_inicio.drop_duplicates().sort_values()
    globais = globais.reindex(calendario_global).ffill()
    fallback_prob = p.semana_inicio.map(globais['_prob'])
    fallback_log = p.semana_inicio.map(globais['_log'])
    p['f_te_mun_prob'] = p['_hist_prob_municipio'].fillna(fallback_prob)
    p['f_te_mun_media_log'] = p['_hist_log_municipio'].fillna(fallback_log)

    pressao = p.assign(_casos_disponiveis=atual).groupby(
        ['uf', 'semana_inicio'], observed=True
    )['_casos_disponiveis']
    p['f_pressao_pe_casos_log'] = np.log1p(
        pressao.transform(lambda s: s.sum(min_count=1)).clip(lower=0)
    )
    p['f_pressao_pe_municipios_ativos'] = pressao.transform(
        lambda s: s.gt(0).sum() / s.notna().sum() if s.notna().any() else np.nan
    )
    p.drop(columns=[
        '_hist_casos_treino', '_hist_prob_treino', '_hist_log_treino',
        '_hist_prob_municipio', '_hist_log_municipio',
    ], inplace=True)

    p['f_incidencia_recente'] = atual / p.populacao * 100000
    p['f_seno_semana'] = np.sin(2 * np.pi * p.numero_semana / 52.1775)
    p['f_cosseno_semana'] = np.cos(2 * np.pi * p.numero_semana / 52.1775)
    p['f_latitude'], p['f_longitude'] = p.latitude, p.longitude
    p['f_log_area'] = np.log1p(p.area_km2)
    p['f_historia_ausente'] = atual.isna().astype('int8')
    p = atributos_vizinhos(cfg, p, atual)
    # Baselines têm a MESMA disponibilidade do candidato, incluindo delay.
    p['baseline_recente'] = atual
    p['baseline_media4'] = grupo_atual.transform(lambda s: s.rolling(4, min_periods=4).mean())
    for h in cfg['treino']['horizontes']:
        p[f'alvo_h{h}'] = g.shift(-h)
        p[f'data_alvo_h{h}'] = p.semana_inicio + pd.to_timedelta(7*h, unit='D')
        # y(t+h-52), não y(t-52): alinha corretamente o baseline por horizonte.
        p[f'baseline_sazonal_h{h}'] = g.shift(52-h)
    features = sorted(c for c in p if c.startswith('f_'))
    for c in features:
        p[c] = pd.to_numeric(p[c], errors='coerce').replace([np.inf, -np.inf], np.nan).astype('float32')
    # Evita replicar dezenas de colunas de identificação/diagnóstico no treino.
    keep = ['codigo_ibge', 'uf', 'porte', 'ano_epi', 'numero_semana', 'semana_inicio',
            'semana_fim', 'populacao', 'casos', 'zero_inferido', 'fonte_recebida', 'semana_madura']
    keep += features + [c for c in p if c.startswith(('alvo_', 'data_alvo_', 'baseline_'))]
    p = p[keep]
    salvar_parquet(p, arquivo(cfg, 'ouro', 'atributos.parquet'))
    salvar_json(arquivo(cfg, 'ouro', 'schema_atributos.json'), {
        'numericas': features, 'categoricas': ['codigo_ibge', 'uf', 'porte'],
        'ausencias_antes_imputacao': {c: float(p[c].isna().mean()) for c in features},
        'cenario': 'retrospectivo_base_revisada; sem snapshots as-of',
        'atraso_casos_semanas': delay,
        'config_preparo': config_preparo(cfg),
    })
    LOG.info('Atributos prontos: %d linhas, %d numéricos + 3 categóricos.', len(p), len(features))
    return p


class ImputadorTreino:
    """Ajuste por fold, serializável; nenhum parâmetro aprende com validação/teste."""

    def fit(self, df, numericas, categoricas):
        self.numericas, self.categoricas = list(numericas), list(categoricas)
        self.ativas = [c for c in numericas if df[c].notna().any()]
        self.excluidas = sorted(set(numericas) - set(self.ativas))
        self.medianas = df[self.ativas].median()
        self.climaticas = [c for c in self.ativas if c.startswith(('f_temperatura', 'f_umidade', 'f_precipitacao'))
                           and not c.endswith(('_ausente', '_ffill'))]
        self.por_uf_semana = df.groupby(['uf', 'numero_semana'], observed=True)[self.climaticas].median()
        self.por_uf = df.groupby('uf', observed=True)[self.climaticas].median()
        self.relatorio = {'ativas': self.ativas, 'excluidas_100pct_ausentes_no_treino': self.excluidas,
                          'medianas_treino': {c: float(v) for c, v in self.medianas.items()},
                          'clima': 'mediana UF/semana do treino -> UF -> global do treino'}
        return self

    def transform(self, df):
        x = df[self.ativas].copy()
        for c in self.climaticas:
            lookup = pd.MultiIndex.from_frame(df[['uf', 'numero_semana']])
            v = self.por_uf_semana[c].reindex(lookup).to_numpy()
            x[c] = x[c].fillna(pd.Series(v, index=x.index))
            x[c] = x[c].fillna(df.uf.map(self.por_uf[c]))
        # Indicador preserva diferença entre dado observado e imputado.
        flags = pd.DataFrame({c + '__ausente': df[c].isna().astype('int8') for c in self.ativas}, index=df.index)
        x = pd.concat([x.fillna(self.medianas), flags], axis=1)
        for c in self.categoricas:
            x[c] = df[c].astype('string').fillna('DESCONHECIDO').astype(str)
        return x


In [ ]:
"""Candidatos adicionais para comparação no mesmo corte temporal.

Modelos tabulares recebem atributos imputados pelo pipeline. ARIMA recebe as
datas e contagens para ajustar uma série temporal agregada por UF.
"""

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler



TIPOS = {'logistica_poisson', 'ebm', 'xgboost', 'lightgbm', 'random_forest',
         'extra_trees', 'bayes_hierarquico', 'nb_misto', 'arima'}


def _amostra(y, limite, semente):
    """Amostragem por classe com pesos inversos para recuperar a prevalência."""
    n = len(y)
    if n <= limite:
        return np.arange(n), np.ones(n)
    rng = np.random.default_rng(semente)
    positivos = np.flatnonzero(y > 0)
    negativos = np.flatnonzero(y == 0)
    n_pos = min(len(positivos), max(1, limite // 2))
    n_neg = min(len(negativos), limite - n_pos)
    if n_neg < 1:
        n_pos, n_neg = limite - 1, 1
    indices = np.sort(np.r_[rng.choice(positivos, n_pos, replace=False),
                            rng.choice(negativos, n_neg, replace=False)])
    pesos = np.where(y[indices] > 0, len(positivos) / n_pos,
                     len(negativos) / n_neg).astype(float)
    pesos /= pesos.mean()
    return indices, pesos


In [ ]:
class ModeloAdicional:
    """Modelo tabular ou temporal com API comum de calibração e inferência."""

    def __init__(self, tipo, cfg):
        if tipo not in TIPOS:
            raise ValueError(f'Modelo desconhecido: {tipo}')
        self.tipo, self.cfg = tipo, cfg

    def _preparar(self, x, numericas, categoricas, y):
        self.numericas = list(numericas)
        self.categoricas = list(categoricas)
        if self.tipo == 'ebm':
            # EBM tem custo próximo de linear no número de atributos. A seleção
            # ocorre só no treino; o código municipal de alta cardinalidade sai.
            ranking = []
            alvo = rotulo_ocorrencia(self.cfg, y).astype(float)
            for c in self.numericas:
                valores = pd.to_numeric(x[c], errors='coerce').to_numpy(float)
                correlacao = np.corrcoef(valores, alvo)[0, 1] if np.std(valores) else 0
                ranking.append((abs(correlacao) if np.isfinite(correlacao) else 0, c))
            ranking.sort(reverse=True)
            self.colunas_ebm = [c for _, c in ranking[:int(self.cfg['treino'].get('max_atributos_ebm', 30))]]
            self.colunas_ebm += [c for c in self.categoricas if c != 'codigo_ibge']
            return self._transformar(x)
        self.codificador = ColumnTransformer([
            ('num', StandardScaler(with_mean=False), self.numericas),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), self.categoricas),
        ], sparse_threshold=1.0)
        return self.codificador.fit_transform(x)

    def _transformar(self, x):
        if self.tipo == 'ebm':
            return x[self.colunas_ebm].copy()
        return self.codificador.transform(x)

    def fit(self, x, y, numericas, categoricas, datas=None):
        y = np.asarray(y, float)
        self.numericas, self.categoricas = list(numericas), list(categoricas)
        if self.tipo == 'arima':
            if datas is None:
                raise ValueError('ARIMA precisa da data de cada observação.')
            self._fit_arima(x, y, datas)
            self.smote = {'habilitado': False, 'motivo': 'modelo_temporal'}
            return self
        limite = int(self.cfg['treino'].get('max_linhas_modelos_adicionais', 120000))
        indices, pesos = _amostra(y, limite, self.cfg['semente'])
        x_fit = x.iloc[indices].reset_index(drop=True)
        y_fit = y[indices]
        self.linhas_usadas = len(indices)
        self.smote = {'habilitado': False, 'sinteticos': 0, 'linhas_treino': len(indices),
                      'amostragem_com_pesos_inversos': len(indices) < len(y)}
        mostrar = self.cfg['treino'].get('progresso', {}).get('mostrar', False)
        if mostrar:
            LOG.info('%s: usando %d de %d linhas para ajuste.', self.tipo, len(indices), len(y))
        if self.tipo == 'bayes_hierarquico':
            if mostrar:
                LOG.info('%s: estimando distribuições por UF e município.', self.tipo)
            self._fit_bayes(x_fit, y_fit, pesos)
            return self
        if self.tipo == 'nb_misto':
            if mostrar:
                LOG.info('%s: ajustando GLM binomial negativo.', self.tipo)
            self._fit_nb(x_fit, y_fit, pesos)
            return self
        xx = self._preparar(x_fit, numericas, categoricas, y_fit)
        if self.tipo == 'logistica_poisson':
            self.classificador = LogisticRegression(max_iter=180, class_weight='balanced',
                                                     solver='liblinear', random_state=self.cfg['semente'])
            self.regressor = PoissonRegressor(alpha=1.0, max_iter=180)
        elif self.tipo == 'ebm':
            try:
                from interpret.glassbox import ExplainableBoostingClassifier, ExplainableBoostingRegressor
            except ImportError as e:
                raise ImportError('Instale interpret-core com requirements.txt para usar ebm.') from e
            args = dict(interactions=0, outer_bags=1, max_rounds=250,
                        n_jobs=self.cfg['threads'], random_state=self.cfg['semente'])
            self.classificador = ExplainableBoostingClassifier(**args)
            self.regressor = ExplainableBoostingRegressor(**args)
        elif self.tipo == 'xgboost':
            try:
                from xgboost import XGBClassifier, XGBRegressor
            except ImportError as e:
                raise ImportError('Instale xgboost com requirements.txt.') from e
            args = dict(n_estimators=180, max_depth=4, learning_rate=0.06, tree_method='hist',
                        n_jobs=self.cfg['threads'], random_state=self.cfg['semente'])
            self.classificador = XGBClassifier(objective='binary:logistic', **args)
            self.regressor = XGBRegressor(objective='count:poisson', **args)
        elif self.tipo == 'lightgbm':
            try:
                from lightgbm import LGBMClassifier, LGBMRegressor
            except ImportError as e:
                raise ImportError('Instale lightgbm com requirements.txt.') from e
            args = dict(n_estimators=180, num_leaves=15, learning_rate=0.06,
                        n_jobs=self.cfg['threads'], random_state=self.cfg['semente'], verbosity=-1)
            self.classificador = LGBMClassifier(objective='binary', **args)
            self.regressor = LGBMRegressor(objective='poisson', **args)
        else:
            klass = RandomForestClassifier if self.tipo == 'random_forest' else ExtraTreesClassifier
            reg = RandomForestRegressor if self.tipo == 'random_forest' else ExtraTreesRegressor
            args = dict(n_estimators=80, max_depth=15, min_samples_leaf=8,
                        max_features=0.7, n_jobs=self.cfg['threads'], random_state=self.cfg['semente'])
            self.classificador, self.regressor = klass(**args), reg(**args)
        peso_pos = float(self.cfg['treino'].get('peso_classe_positiva', 4.0))
        eventos = rotulo_ocorrencia(self.cfg, y_fit)
        peso_classe = pesos * np.where(eventos == 1, peso_pos, 1.0)
        if mostrar:
            LOG.info('%s: ajustando classificador de ocorrência.', self.tipo)
        self.classificador.fit(xx, eventos, sample_weight=peso_classe)
        positivos = eventos == 1
        if mostrar:
            LOG.info('%s: ajustando regressor de contagem (%d positivos).', self.tipo, int(positivos.sum()))
        self.regressor.fit(xx[positivos], y_fit[positivos], sample_weight=pesos[positivos])
        return self

    def _fit_arima(self, x, y, datas):
        try:
            from statsmodels.tsa.arima.model import ARIMA
        except ImportError as erro:
            raise ImportError('Instale statsmodels com requirements.txt para usar ARIMA.') from erro
        datas = pd.DatetimeIndex(pd.to_datetime(datas, errors='raise'))
        if datas.isna().any() or not np.isfinite(y).all() or (y < 0).any():
            raise ValueError('ARIMA requer datas válidas e contagens finitas não negativas.')
        if not {'uf', 'populacao'}.issubset(x.columns):
            raise ValueError('ARIMA requer as colunas uf e populacao.')

        ordem = tuple(self.cfg['treino'].get('arima', {}).get('ordem', [1, 1, 1]))
        minimo = int(self.cfg['treino'].get('arima', {}).get('min_semanas', 52))
        dados = pd.DataFrame({
            'uf': x.uf.astype(str).to_numpy(),
            'data': datas.to_numpy(),
            'casos': y,
            'populacao': pd.to_numeric(x.populacao, errors='raise').to_numpy(float),
        })
        if (dados.populacao <= 0).any():
            raise ValueError('ARIMA requer população positiva.')

        self.arima_por_uf = {}
        for uf, grupo in dados.groupby('uf', sort=True):
            semanal = grupo.groupby('data', sort=True).agg(
                casos=('casos', 'sum'), populacao=('populacao', 'sum'))
            if len(semanal) < minimo:
                raise ValueError(f'ARIMA da UF {uf} requer ao menos {minimo} semanas; recebeu {len(semanal)}.')
            grade = pd.date_range(semanal.index.min(), semanal.index.max(), freq='7D')
            incidencia = (semanal.casos / semanal.populacao * 100000).reindex(grade)
            ajustado = ARIMA(incidencia, order=ordem, missing='none').fit()
            self.arima_por_uf[uf] = {
                'modelo': ajustado,
                'ultima_data': grade[-1],
            }
        self.arima_ordem = ordem
        self.arima_semanas_treino = int(len(dados))

    def _fit_bayes(self, x, y, pesos):
        # Gamma-Poisson hierárquico empírico: média estadual informa o prior
        # da taxa municipal. É um baseline de contagem, sem efeitos climáticos.
        uf = x['uf'].astype(str).to_numpy()
        municipio = x['codigo_ibge'].astype(str).to_numpy()
        self.media_global = float(np.average(y, weights=pesos))
        self.medias_uf = {}
        self.medias_municipio = {}
        for estado in np.unique(uf):
            mask = uf == estado
            media_uf = float(np.average(y[mask], weights=pesos[mask]))
            self.medias_uf[estado] = media_uf
            for cod in np.unique(municipio[mask]):
                m = mask & (municipio == cod)
                # força anterior de 52 semanas; contribuição municipal posterior.
                self.medias_municipio[cod] = float((np.sum(pesos[m] * y[m]) + 52 * media_uf) /
                                                   (np.sum(pesos[m]) + 52))

    def _fit_nb(self, x, y, pesos):
        # NB GLM com efeito fixo de UF e correção municipal regularizada.
        # Em dados massivos, evita matriz densa com um parâmetro por município.
        try:
            import statsmodels.api as sm
        except ImportError as e:
            raise ImportError('Instale statsmodels com requirements.txt.') from e
        self.colunas_nb = [c for c in self.numericas if c.startswith('f_casos_')][:12]
        self.colunas_nb += [c for c in ('f_log_populacao', 'f_densidade') if c in self.numericas]
        self.uf_nb = sorted(x.uf.astype(str).unique())
        matriz = self._matriz_nb(x)
        media, variancia = float(np.average(y, weights=pesos)), float(np.average((y - np.average(y, weights=pesos)) ** 2, weights=pesos))
        self.alpha_nb = max(0.01, (variancia - media) / max(media * media, 0.01))
        self.regressor = sm.GLM(y, matriz, family=sm.families.NegativeBinomial(alpha=self.alpha_nb),
                                freq_weights=pesos).fit(maxiter=60, disp=0)
        mu = np.maximum(self.regressor.predict(matriz), 1e-4)
        self.efeito_municipio = {}
        codigos = x.codigo_ibge.astype(str).to_numpy()
        for cod in np.unique(codigos):
            m = codigos == cod
            self.efeito_municipio[cod] = float((np.sum(pesos[m] * y[m]) + 52 * np.mean(mu[m])) /
                                                (np.sum(pesos[m] * mu[m]) + 52 * np.mean(mu[m])))

    def _matriz_nb(self, x):
        numerico = x[self.colunas_nb].to_numpy(float)
        escala = getattr(self, 'escala_nb', None)
        if escala is None:
            self.escala_nb = np.maximum(np.std(numerico, axis=0), 1e-6)
        numerico = np.clip(numerico / self.escala_nb, -20, 20)
        uf = x.uf.astype(str).to_numpy()
        dummies = np.column_stack([uf == c for c in self.uf_nb[1:]]).astype(float) if len(self.uf_nb) > 1 else np.empty((len(x), 0))
        return np.column_stack([np.ones(len(x)), numerico, dummies])

    def predict(self, x, datas=None):
        if self.tipo == 'arima':
            if datas is None:
                raise ValueError('ARIMA precisa da data de cada previsão.')
            datas = pd.DatetimeIndex(pd.to_datetime(datas, errors='raise'))
            pred = np.empty(len(x), dtype=float)
            ufs = x.uf.astype(str).to_numpy()
            populacao = pd.to_numeric(x.populacao, errors='raise').to_numpy(float)
            for uf in np.unique(ufs):
                if uf not in self.arima_por_uf:
                    raise ValueError(f'ARIMA não foi ajustado para a UF {uf}.')
                mask = ufs == uf
                ajuste = self.arima_por_uf[uf]
                passos = ((datas[mask] - ajuste['ultima_data']).days / 7).astype(int)
                if (passos < 1).any() or not np.allclose(
                        (datas[mask] - ajuste['ultima_data']).days / 7, passos):
                    raise ValueError(f'ARIMA recebeu datas não futuras ou desalinhadas da semana para UF {uf}.')
                previsoes = ajuste['modelo'].forecast(steps=int(passos.max()))
                pred[mask] = (np.asarray(previsoes)[passos - 1]
                              * populacao[mask] / 100000)
            return np.maximum(pred, 0.)
        if self.tipo == 'bayes_hierarquico':
            uf, cod = x.uf.astype(str).to_numpy(), x.codigo_ibge.astype(str).to_numpy()
            mu = np.array([self.medias_municipio.get(c, self.medias_uf.get(u, self.media_global))
                           for c, u in zip(cod, uf)])
        elif self.tipo == 'nb_misto':
            mu = self.regressor.predict(self._matriz_nb(x))
            mu *= np.array([self.efeito_municipio.get(c, 1.) for c in x.codigo_ibge.astype(str)])
        else:
            xx = self._transformar(x)
            mu = self.regressor.predict(xx) * self.classificador.predict_proba(xx)[:, 1]
        return np.maximum(np.asarray(mu, float), 0.)

    def predict_ocorrencia(self, x, datas=None):
        if self.tipo == 'arima':
            return 1 - np.exp(-self.predict(x, datas))
        if self.tipo == 'bayes_hierarquico':
            return 1 - np.exp(-self.predict(x))
        if self.tipo == 'nb_misto':
            mu = self.predict(x)
            return 1 - np.power(1 + self.alpha_nb * mu, -1 / self.alpha_nb)
        return self.classificador.predict_proba(self._transformar(x))[:, 1]

    def explicacao(self):
        """Resumo global auditável da parte de ocorrência ou da taxa prevista."""
        if self.tipo == 'arima':
            return {
                'metodo': 'ARIMA_por_UF_em_incidencia_semanal; escala_por_populacao_municipal',
                'ordem': list(self.arima_ordem),
                'semanas_treino': self.arima_semanas_treino,
                'ufs': sorted(self.arima_por_uf),
            }
        if self.tipo == 'bayes_hierarquico':
            return {'metodo': 'posterior_Gamma_Poisson_por_UF_e_municipio',
                    'media_global': self.media_global, 'medias_uf': self.medias_uf,
                    'forca_prior_semanas': 52}
        if self.tipo == 'nb_misto':
            nomes = ['intercepto'] + self.colunas_nb + [f'uf_{u}' for u in self.uf_nb[1:]]
            valores = np.asarray(self.regressor.params, float)
            metodo = 'coeficientes_log_taxa_NB_e_correcao_municipal'
        elif self.tipo == 'ebm':
            nomes = list(self.classificador.term_names_)
            valores = np.asarray(self.classificador.term_importances(), float)
            metodo = 'importancia_termos_EBM_ocorrencia'
        elif self.tipo == 'logistica_poisson':
            nomes = list(self.codificador.get_feature_names_out())
            valores = np.asarray(self.classificador.coef_[0], float)
            metodo = 'coeficientes_log_odds_ocorrencia'
        else:
            nomes = list(self.codificador.get_feature_names_out())
            valores = np.asarray(self.classificador.feature_importances_, float)
            metodo = 'importancia_estrutura_arvores_ocorrencia'
        ordem = np.argsort(-np.abs(valores))[:30]
        return {'metodo': metodo, 'top_atributos': [
            {'atributo': str(nomes[i]), 'valor': float(valores[i])}
            for i in ordem if np.isfinite(valores[i])]}


In [ ]:
import json
import shutil
import copy
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             mean_absolute_error, mean_squared_error,
                             precision_recall_curve, precision_recall_fscore_support)



def smote_baixo(cfg, x, y, categoricas):
    """SMOTENC pequeno, por UF e somente no treino do classificador Hurdle."""
    # `y` é posicional. Os folds preservam o índice do painel original, então
    # normalizamos antes de separar as UFs para não desalinhá-los.
    x = x.reset_index(drop=True)
    conf = cfg['treino'].get('smote', {})
    y = np.asarray(y, dtype=int)
    antes = {'linhas': int(len(y)), 'positivos': int(y.sum()), 'negativos': int((y == 0).sum())}
    if not conf.get('habilitado', False):
        return x, y, {**antes, 'habilitado': False, 'sinteticos': 0}
    try:
        from imblearn.over_sampling import SMOTENC
    except ImportError as erro:
        raise ImportError('Instale imbalanced-learn: python -m pip install -r treinamentoDoModelo/requirements.txt') from erro
    alvo = float(conf['razao_pos_neg_max'])
    if not 0 < alvo <= 1:
        raise ValueError('smote.razao_pos_neg_max deve estar entre 0 e 1.')
    k = int(conf['k_vizinhos'])
    if k < 1:
        raise ValueError('smote.k_vizinhos deve ser >= 1.')
    cat_idx = [x.columns.get_loc(c) for c in categoricas]
    partes_x, partes_y, grupos = [], [], {}
    # Uma UF não mistura padrões climáticos/territoriais diretamente com outra.
    for i, (uf, grupo) in enumerate(x.groupby('uf', sort=True, observed=True)):
        labels = y[grupo.index.to_numpy()]
        positivos, negativos = int(labels.sum()), int((labels == 0).sum())
        desejado = int(np.ceil(alvo * negativos))
        adicionar = min(max(0, desejado - positivos), int(conf['max_sinteticos_por_uf']))
        if adicionar == 0 or positivos < k + 1 or negativos == 0:
            partes_x.append(grupo)
            partes_y.append(labels)
            grupos[str(uf)] = {'originais': len(grupo), 'positivos': positivos, 'sinteticos': 0}
            continue
        estrategia = {1: positivos + adicionar}
        sampler = SMOTENC(categorical_features=cat_idx, sampling_strategy=estrategia,
                           k_neighbors=k, random_state=cfg['semente'] + i)
        # DataFrame preserva os nomes; categorias nunca são interpoladas como números.
        gx, gy = sampler.fit_resample(grupo, labels)
        gx = pd.DataFrame(gx, columns=x.columns)
        for c in categoricas:
            gx[c] = gx[c].astype(str)
        for c in x.columns:
            if c not in categoricas:
                gx[c] = pd.to_numeric(gx[c], errors='raise')
        partes_x.append(gx)
        partes_y.append(np.asarray(gy, dtype=int))
        grupos[str(uf)] = {'originais': len(grupo), 'positivos': positivos,
                            'sinteticos': int(len(gy)-len(labels))}
    out_x = pd.concat(partes_x, ignore_index=True)
    out_y = np.concatenate(partes_y)
    return out_x, out_y, {**antes, 'habilitado': True, 'sinteticos': int(len(out_y)-len(y)),
                           'depois_linhas': int(len(out_y)), 'depois_positivos': int(out_y.sum()),
                           'por_uf': grupos, 'razao_maxima': alvo, 'k_vizinhos': k,
                           'observacao': 'SMOTENC só no classificador; regressão usa positivos reais.'}


def metricas(y, pred, pop=None):
    y, pred = np.asarray(y, dtype=float), np.maximum(np.asarray(pred, dtype=float), 0)
    if not len(y):
        return {'n': 0}
    result = {'n': len(y), 'mae': float(mean_absolute_error(y, pred)),
              'rmse': float(np.sqrt(mean_squared_error(y, pred))),
              'wape': float(np.abs(y-pred).sum()/y.sum()) if y.sum() > 0 else None,
              'rmsle': float(np.sqrt(np.mean((np.log1p(y)-np.log1p(pred))**2)))}
    if pop is not None:
        pop = np.asarray(pop, dtype=float)
        ok = np.isfinite(pop) & (pop > 0)
        result['mae_incidencia_100k'] = float(np.mean(np.abs(y[ok]-pred[ok])/pop[ok]*100000)) if ok.any() else None
    return result


def avaliar(df, y, pred, lower=None, upper=None, threshold=None, prob=None):
    y, pred = np.asarray(y, float), np.maximum(np.asarray(pred, float), 0)
    out = {'global': metricas(y, pred, df.populacao)}
    tmp = df[['codigo_ibge', 'uf', 'porte', 'populacao']].copy()
    tmp['y'], tmp['pred'] = y, pred
    out['macro_mae'] = float(pd.Series(np.abs(y-pred), index=tmp.index).groupby(tmp.codigo_ibge, observed=True).mean().mean())
    out['por_uf'], out['por_porte'] = {}, {}
    for coluna, destino in [('uf', 'por_uf'), ('porte', 'por_porte')]:
        for key, g in tmp.groupby(coluna, observed=True):
            out[destino][str(key)] = metricas(g.y, g.pred, g.populacao)
    out['semanas_zero'] = metricas(y[y == 0], pred[y == 0])
    out['semanas_positivas'] = metricas(y[y > 0], pred[y > 0])
    if lower is not None:
        out['intervalo'] = {'cobertura': float(np.mean((y >= lower) & (y <= upper))),
                           'largura_media': float(np.mean(upper-lower))}
    if threshold is not None:
        actual = y > threshold
        forecast = prob >= 0.5 if prob is not None else pred > threshold
        precision, recall, f1, _ = precision_recall_fscore_support(actual, forecast, average='binary', zero_division=0)
        out['alerta_experimental'] = {'precision': float(precision), 'recall': float(recall), 'f1': float(f1),
                                     'falsos_positivos': int(np.sum(forecast & ~actual)),
                                     'positivos': int(actual.sum())}
        if prob is not None:
            out['alerta_experimental']['brier'] = float(brier_score_loss(actual, prob))
            out['alerta_experimental']['pr_auc'] = float(average_precision_score(actual, prob)) if actual.any() else None
    return out


def rotulo_ocorrencia(cfg, y):
    minimo = float(cfg['treino'].get('ocorrencia', {}).get('casos_minimos', 1))
    return (np.asarray(y, dtype=float) >= minimo).astype(int)


def metricas_ocorrencia(y, prob, limiar=0.5, casos_minimos=1):
    """Métricas do evento configurado, sem calibrar no período de validação."""
    y = (np.asarray(y, dtype=float) >= float(casos_minimos)).astype(int)
    prob = np.clip(np.asarray(prob, dtype=float), 0, 1)
    previsto = prob >= limiar
    precision, recall, f1, _ = precision_recall_fscore_support(
        y, previsto, average='binary', zero_division=0)
    return {
        'positivos': int(y.sum()),
        'prevalencia': float(y.mean()) if len(y) else None,
        'precision': float(precision), 'recall': float(recall), 'f1': float(f1),
        'pr_auc': float(average_precision_score(y, prob)) if y.any() else None,
        'brier': float(brier_score_loss(y, prob)) if len(y) else None,
    }


In [ ]:
def calibrar_limiar_ocorrencia(cfg, modelo, dados, y):
    """Escolhe o menor corte que mantém a precisão mínima no bloco de calibração."""
    conf = cfg['treino'].get('ocorrencia', {})
    piso = float(conf.get('precisao_minima', 0.35))
    padrao = float(conf.get('limiar_padrao', 0.5))
    recall_minimo = float(conf.get('recall_minimo', 0.80))
    atual = rotulo_ocorrencia(cfg, y)
    prob = modelo.predict_ocorrencia(dados)
    if not atual.any():
        modelo.limiar_ocorrencia = padrao
        modelo.calibracao_ocorrencia = {'motivo': 'calibracao_sem_positivos', 'limiar': padrao}
        return
    precision, recall, limiares = precision_recall_curve(atual, prob)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    candidatos = np.flatnonzero((precision[:-1] >= piso) & (recall[:-1] >= recall_minimo))
    if len(candidatos):
        i = candidatos[np.argmax(f1[candidatos])]
    else:
        candidatos = np.flatnonzero(precision[:-1] >= piso)
        if len(candidatos):
            i = candidatos[np.argmax(f1[candidatos])]
        else:
            i = int(np.argmax(f1))
    corte = float(limiares[i]) if len(limiares) else padrao
    modelo.limiar_ocorrencia = corte
    modelo.calibracao_ocorrencia = {
        'limiar': corte, 'precisao_minima': piso, 'recall_minimo': recall_minimo,
        'precision_calibracao': float(precision[i]),
        'recall_calibracao': float(recall[i]),
    }


In [ ]:
class Candidato:
    def __init__(self, cfg, tipo):
        self.cfg, self.tipo = cfg, tipo

    def _parametros(self, iterations):
        t = self.cfg['treino']
        progresso = t.get('progresso', {})
        return dict(iterations=iterations, depth=t['depth'], learning_rate=t['learning_rate'],
                    l2_leaf_reg=t['l2_leaf_reg'], thread_count=self.cfg['threads'],
                    random_seed=self.cfg['semente'], allow_writing_files=False,
                    verbose=int(progresso.get('intervalo_arvores', 50)) if progresso.get('mostrar') else False)

    def fit(self, df, y, numericas, categoricas, data_alvo=None):
        if self.tipo == 'arima':
            if data_alvo is None:
                raise ValueError('ARIMA precisa da coluna com as datas dos alvos.')
            self.data_alvo = data_alvo
            self.adicional = ModeloAdicional(self.tipo, self.cfg).fit(
                df[['uf', 'populacao']], np.asarray(y, float), [], [],
                datas=df[data_alvo])
            self.smote = self.adicional.smote
            self.iterations = None
            return self
        if self.tipo in TIPOS_ADICIONAIS:
            self.imputador = ImputadorTreino().fit(df, numericas, categoricas)
            x_adicional = self.imputador.transform(df)
            self.adicional = ModeloAdicional(self.tipo, self.cfg).fit(
                x_adicional, np.asarray(y, float),
                [c for c in x_adicional if c not in categoricas], categoricas)
            self.smote = self.adicional.smote
            self.iterations = None
            return self
        # Cauda interna para early stopping. A calibração externa fica intocada.
        datas = np.sort(df.semana_inicio.unique())
        if len(datas) < 30:
            raise ValueError('Treino insuficiente: precisa de >=30 semanas distintas.')
        boundary = datas[-min(13, len(datas)//4)]
        prefixo = df.semana_inicio < boundary
        self.imputador = ImputadorTreino().fit(df[prefixo], numericas, categoricas)
        x = self.imputador.transform(df)
        y = np.asarray(y, dtype=float)
        params = self._parametros(self.cfg['treino']['iterations'])
        if self.cfg['treino'].get('treinamento_rapido', False):
            # Um ajuste completo por candidato/fold. Evita o piloto + ajuste final,
            # que dobrava o trabalho para escolher o número de árvores.
            iterations = int(self.cfg['treino']['iterations'])
        else:
            piloto = CatBoostRegressor(loss_function='Poisson', **params)
            piloto.fit(x[prefixo], y[prefixo], cat_features=categoricas,
                       eval_set=(x[~prefixo], y[~prefixo]),
                       early_stopping_rounds=self.cfg['treino']['early_stopping_rounds'])
            iterations = max(1, piloto.tree_count_)
        self.imputador = ImputadorTreino().fit(df, numericas, categoricas)
        x = self.imputador.transform(df)
        params = self._parametros(iterations)
        if self.tipo == 'poisson':
            if self.cfg['treino'].get('progresso', {}).get('mostrar'):
                LOG.info('CatBoost Poisson: ajustando regressor de contagem (%d linhas).', len(x))
            self.regressor = CatBoostRegressor(loss_function='Poisson', **params)
            self.regressor.fit(x, y, cat_features=categoricas)
        elif self.tipo == 'hurdle':
            eventos = rotulo_ocorrencia(self.cfg, y)
            positivos = eventos == 1
            if eventos.sum() < 20 or (eventos == 0).sum() < 20:
                raise ValueError('Hurdle precisa de pelo menos 20 eventos positivos e 20 negativos no treino.')
            x_smote, y_smote, self.smote = smote_baixo(self.cfg, x, eventos, categoricas)
            peso = float(self.cfg['treino'].get('peso_classe_positiva', 1.0))
            if peso < 1:
                raise ValueError('peso_classe_positiva deve ser >= 1.')
            params_classificador = dict(params)
            if peso != 1:
                params_classificador['class_weights'] = [1.0, peso]
            self.classificador = CatBoostClassifier(loss_function='Logloss', **params_classificador)
            if self.cfg['treino'].get('progresso', {}).get('mostrar'):
                LOG.info('CatBoost Hurdle: ajustando classificador (%d linhas).', len(x_smote))
            self.classificador.fit(x_smote, y_smote, cat_features=categoricas)
            self.smote['peso_classe_positiva'] = peso
            if self.cfg['treino'].get('progresso', {}).get('mostrar'):
                LOG.info('CatBoost Hurdle: ajustando regressor (%d semanas positivas).', int(positivos.sum()))
            self.regressor = CatBoostRegressor(loss_function='Poisson', **params)
            self.regressor.fit(x[positivos], y[positivos], cat_features=categoricas)
        else:
            raise ValueError(f'Candidato desconhecido {self.tipo}')
        self.iterations = iterations
        if self.tipo == 'poisson':
            self.smote = {'habilitado': False, 'motivo': 'modelo_de_contagem_sem_classificador'}
        return self

    def predict(self, df):
        if self.tipo == 'arima':
            pred = self.adicional.predict(df[['uf', 'populacao']], df[self.data_alvo])
            if not np.isfinite(pred).all():
                raise ValueError('Previsão ARIMA não finita.')
            return np.maximum(pred, 0.)
        x = self.imputador.transform(df)
        if self.tipo in TIPOS_ADICIONAIS:
            pred = self.adicional.predict(x)
            if not np.isfinite(pred).all():
                raise ValueError('Previsão não finita. Modelo não pode ser publicado.')
            return pred
        # CatBoost Poisson predict padrão é log(mu); Exponent fornece mu.
        pred = self.regressor.predict(x, prediction_type='Exponent')
        if self.tipo == 'hurdle':
            pred *= self.classificador.predict_proba(x)[:, 1]
        if not np.isfinite(pred).all():
            raise ValueError('Previsão não finita. Modelo não pode ser publicado.')
        return np.maximum(pred, 0.)

    def predict_ocorrencia(self, df):
        if self.tipo == 'arima':
            return self.adicional.predict_ocorrencia(
                df[['uf', 'populacao']], df[self.data_alvo])
        if self.tipo in TIPOS_ADICIONAIS:
            return self.adicional.predict_ocorrencia(self.imputador.transform(df))
        if self.tipo != 'hurdle':
            raise ValueError('Probabilidade de ocorrência só existe no modelo Hurdle.')
        return self.classificador.predict_proba(self.imputador.transform(df))[:, 1]

    def explicacao(self):
        if self.tipo in TIPOS_ADICIONAIS:
            return self.adicional.explicacao()
        if self.tipo == 'hurdle':
            valores = self.classificador.get_feature_importance()
            nomes = self.classificador.feature_names_
        else:
            valores = self.regressor.get_feature_importance()
            nomes = self.regressor.feature_names_
        ordem = np.argsort(-np.abs(valores))[:30]
        return {'metodo': 'importancia_CatBoost', 'top_atributos': [
            {'atributo': str(nomes[i]), 'valor': float(valores[i])} for i in ordem]}


def ajustar_calibracao(cfg, modelo, dados, y):
    pred = modelo.predict(dados)
    residuo = np.asarray(y) - pred
    alpha = cfg['treino']['cobertura_intervalo']
    def quantil(r):
        nivel = min(1., np.ceil((len(r)+1)*alpha)/len(r))
        return float(np.quantile(np.abs(r), nivel, method='higher'))
    if len(residuo) < 30:
        raise ValueError('Calibração insuficiente: menos de 30 linhas.')
    modelo.calibracao = {'global': quantil(residuo), 'por_uf': {}, 'residuos': {}}
    modelo.calibracao['residuos']['global'] = np.sort(residuo)
    for uf in dados.uf.unique():
        r = residuo[dados.uf.to_numpy() == uf]
        if len(r) >= 100:
            modelo.calibracao['por_uf'][str(uf)] = quantil(r)
            modelo.calibracao['residuos'][str(uf)] = np.sort(r)


In [ ]:
def prever_com_intervalos(modelo, dados, threshold):
    pred = modelo.predict(dados)
    q = dados.uf.astype(str).map(modelo.calibracao['por_uf']).fillna(modelo.calibracao['global']).to_numpy()
    prob = np.zeros(len(dados))
    for uf in dados.uf.astype(str).unique():
        mask = dados.uf.astype(str).to_numpy() == uf
        resid = modelo.calibracao['residuos'].get(uf, modelo.calibracao['residuos']['global'])
        # Distribuição residual empírica; checar calibração prospectiva depois.
        prob[mask] = 1 - np.searchsorted(resid, threshold[mask] - pred[mask], side='right') / len(resid)
    return pred, np.maximum(0., pred-q), pred+q, prob


def limiares(cfg, treino, y):
    d = treino[['codigo_ibge', 'uf']].copy()
    d['y'] = np.asarray(y)
    q = cfg['treino']['alerta_quantil']
    minimo = cfg['treino']['alerta_min_casos']
    return {'municipio': d.groupby('codigo_ibge', observed=True).y.quantile(q).clip(lower=minimo).to_dict(),
            'uf': d.groupby('uf', observed=True).y.quantile(q).clip(lower=minimo).to_dict(),
            'global': float(max(minimo, d.y.quantile(q)))}


def aplicar_limiar(dados, limites):
    return dados.codigo_ibge.map(limites['municipio']).fillna(dados.uf.map(limites['uf'])).fillna(limites['global']).to_numpy(float)


def carregar_dados(cfg):
    p = pd.read_parquet(arquivo(cfg, 'ouro', 'atributos.parquet'))
    for c in ['codigo_ibge', 'uf', 'porte']:
        p[c] = p[c].astype(str)
    schema = json.loads(arquivo(cfg, 'ouro', 'schema_atributos.json').read_text())
    preparo_salvo = dict(schema.get('config_preparo', {}))
    preparo_esperado = config_preparo(cfg)
    # Um painel preparado com h1..h4 também contém tudo de que h1/h2 precisa.
    # Permitir apenas esse subconjunto evita reconstruir atributos para o treino rápido.
    horizontes_salvos = set(preparo_salvo.pop('horizontes', []))
    horizontes_esperados = set(preparo_esperado.pop('horizontes', []))
    reserva_salva = preparo_salvo.get('reserva_espacial', {})
    reserva_esperada = preparo_esperado.get('reserva_espacial', {})
    # Uma reserva calculada antes da primeira validação atual é causalmente segura.
    # Reutilizá-la evita refazer o painel só porque se removeu um fold antigo.
    if (reserva_salva.get('fracao') == reserva_esperada.get('fracao') and
            reserva_salva.get('ano_estratificacao', float('inf')) <= reserva_esperada.get('ano_estratificacao', -float('inf'))):
        preparo_esperado['reserva_espacial'] = reserva_salva
    if not horizontes_esperados.issubset(horizontes_salvos) or preparo_salvo != preparo_esperado:
        raise ValueError('Configuração de dados mudou. Execute preparar/atributos com esta configuração antes de treinar.')
    reservados = set(json.loads(arquivo(cfg, 'ouro', 'municipios_reservados.json').read_text()))
    return p, schema, reservados


def elegiveis(cfg, dados, h):
    mais_antigo = cfg['atributos']['atraso_casos_semanas'] + max(cfg['atributos']['lags'])
    return (dados[f'alvo_h{h}'].notna() & dados[f'f_casos_lag{mais_antigo}'].notna()
            & (dados.populacao.isna() | dados.populacao.gt(0)) & dados.baseline_recente.notna()
            & dados.baseline_media4.notna() & dados[f'baseline_sazonal_h{h}'].notna())


def separar_ajuste_calibracao(cfg, dados, fim, h):
    # Purga por DATA DO ALVO, não só por origem. Nenhum rótulo atravessa corte.
    dados = dados[elegiveis(cfg, dados, h) & (dados[f'data_alvo_h{h}'] < fim)]
    inicio_cal = fim - pd.Timedelta(weeks=cfg['treino']['calibracao_semanas'])
    ajuste = dados[dados[f'data_alvo_h{h}'] < inicio_cal]
    cal = dados[dados[f'data_alvo_h{h}'] >= inicio_cal]
    if ajuste.empty or cal.empty:
        raise ValueError(f'Sem treino/calibração antes de {fim.date()}. Verifique recorte/anos.')
    return ajuste, cal


def _config_tentativa(cfg, tentativa):
    """Cria uma configuração isolada: a busca nunca altera config.yaml."""
    novo = copy.deepcopy(cfg)
    treino = novo['treino']
    for chave in ('iterations', 'depth', 'learning_rate', 'l2_leaf_reg',
                  'peso_classe_positiva'):
        if chave in tentativa:
            treino[chave] = tentativa[chave]
    if 'smote' in tentativa:
        treino['smote'].update(tentativa['smote'])
    treino['candidatos'] = ['hurdle']
    return novo


def _resumo_busca(registros, duracao):
    ocorrencia = [r['ocorrencia'] for r in registros]
    regressao = [r['regressao'] for r in registros]
    def media(chave, itens):
        valores = [x[chave] for x in itens if x.get(chave) is not None]
        return float(np.mean(valores)) if valores else None
    return {
        'avaliacoes': len(registros), 'segundos': round(float(duracao), 2),
        'macro_mae': media('macro_mae', regressao),
        'pr_auc_ocorrencia': media('pr_auc', ocorrencia),
        'recall_ocorrencia': media('recall', ocorrencia),
        'f1_ocorrencia': media('f1', ocorrencia),
        'precision_ocorrencia': media('precision', ocorrencia),
    }


def _ordem_busca(resultado):
    """Alta sensibilidade primeiro; qualidade de ranking e contagem desempata."""
    r = resultado['resumo']
    return (-(r['recall_ocorrencia'] if r['recall_ocorrencia'] is not None else -1),
            -(r['pr_auc_ocorrencia'] if r['pr_auc_ocorrencia'] is not None else -1),
            r['macro_mae'] if r['macro_mae'] is not None else float('inf'),
            r['segundos'])


def _avaliar_tentativa(cfg, dados, schema, reservados, anos, horizontes):
    """Backtest temporal enxuto; não calibra nem abre os anos de teste."""
    inicio = time.perf_counter()
    conhecidos = ~dados.codigo_ibge.isin(reservados)
    registros = []
    for h in horizontes:
        for ano in anos:
            fim, limite = pd.Timestamp(inicio_ano_epi(ano)), pd.Timestamp(inicio_ano_epi(ano + 1))
            fit, cal = separar_ajuste_calibracao(cfg, dados[conhecidos], fim, h)
            val = dados[elegiveis(cfg, dados, h) & dados[f'data_alvo_h{h}'].ge(fim)
                        & dados[f'data_alvo_h{h}'].lt(limite) & conhecidos]
            if val.empty:
                raise ValueError(f'Sem linhas de validação em {ano}/h{h}.')
            modelo = Candidato(cfg, 'hurdle').fit(
                fit, fit[f'alvo_h{h}'], schema['numericas'], schema['categoricas'],
                data_alvo=f'data_alvo_h{h}')
            calibrar_limiar_ocorrencia(cfg, modelo, cal, cal[f'alvo_h{h}'])
            pred = modelo.predict(val)
            registros.append({
                'ano': int(ano), 'horizonte': int(h), 'linhas_treino': int(len(fit)),
                'linhas_validacao': int(len(val)),
                'regressao': avaliar(val, val[f'alvo_h{h}'], pred),
                'ocorrencia': metricas_ocorrencia(val[f'alvo_h{h}'], modelo.predict_ocorrencia(val),
                                                   modelo.limiar_ocorrencia, float(cfg['treino']['ocorrencia'].get('casos_minimos', 1))),
                'smote': modelo.smote,
            })
    duracao = time.perf_counter() - inicio
    return registros, _resumo_busca(registros, duracao)


In [ ]:
def buscar_hiperparametros(cfg):
    """Grade curta com seleção temporal da classe minoritária e promoção de finalistas."""
    busca = cfg['treino'].get('busca_hiperparametros', {})
    if not busca.get('habilitado', False):
        raise ValueError('Habilite treino.busca_hiperparametros no config.yaml para executar a busca.')
    testes = set(cfg['treino']['testes_anos'])
    triagem_anos = list(busca['anos_triagem'])
    confirmacao_anos = list(busca['anos_confirmacao'])
    if set(triagem_anos + confirmacao_anos) & testes:
        raise ValueError('A busca de hiperparâmetros não pode abrir os anos de teste bloqueados.')
    dados, schema, reservados = carregar_dados(cfg)
    pasta = arquivo(cfg, 'relatorios', 'busca_hiperparametros') / \
        datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    pasta.mkdir(parents=True)
    triagem = []
    for tentativa in busca['tentativas']:
        nome = tentativa['nome']
        LOG.info('Busca temporal — triagem: %s.', nome)
        cfg_tentativa = _config_tentativa(cfg, tentativa)
        registros, resumo = _avaliar_tentativa(
            cfg_tentativa, dados, schema, reservados, triagem_anos, busca['horizontes_triagem'])
        triagem.append({'nome': nome, 'tentativa': tentativa, 'resumo': resumo, 'registros': registros})
    triagem.sort(key=_ordem_busca)
    salvar_json(pasta / 'triagem.json', triagem)
    finalistas = triagem[:int(busca['finalistas'])]
    confirmacao = []
    for resultado in finalistas:
        LOG.info('Busca temporal — confirmação: %s.', resultado['nome'])
        cfg_tentativa = _config_tentativa(cfg, resultado['tentativa'])
        registros, resumo = _avaliar_tentativa(
            cfg_tentativa, dados, schema, reservados, confirmacao_anos,
            busca['horizontes_confirmacao'])
        confirmacao.append({'nome': resultado['nome'], 'tentativa': resultado['tentativa'],
                            'resumo': resumo, 'registros': registros})
    confirmacao.sort(key=_ordem_busca)
    vencedor = confirmacao[0]
    cfg_final = _config_tentativa(cfg, vencedor['tentativa'])
    # H1/H2 foram os únicos horizontes que superaram baselines no experimento anterior.
    # A configuração exportada evita voltar a gastar tempo com H3/H4 por padrão.
    cfg_final['treino']['horizontes'] = list(busca['horizontes_confirmacao'])
    salvar_json(pasta / 'confirmacao.json', confirmacao)
    salvar_json(pasta / 'melhor_config_busca.json', cfg_final)
    salvar_json(pasta / 'resultado.json', {
        'vencedor': vencedor['nome'], 'criterio': [
            'maior recall de ocorrência (y>0), com precisão mínima calibrada',
            'maior PR-AUC de ocorrência', 'menor MAE macro municipal', 'menor duração'],
        'triagem_anos': triagem_anos, 'confirmacao_anos': confirmacao_anos,
        'horizontes_triagem': busca['horizontes_triagem'],
        'horizontes_confirmacao': busca['horizontes_confirmacao'],
        'config_para_treino': str(pasta / 'melhor_config_busca.json'),
    })
    LOG.info('Busca concluída. Vencedor: %s. Configuração: %s', vencedor['nome'], pasta / 'melhor_config_busca.json')
    return pasta


In [ ]:
def treinar(cfg):
    dados, schema, reservados = carregar_dados(cfg)
    conhecidos = ~dados.codigo_ibge.isin(reservados)
    pasta = arquivo(cfg, 'modelos', datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    pasta.mkdir(parents=True)
    salvar_json(pasta / 'config.json', cfg)
    salvar_json(pasta / 'ambiente.json', ambiente())
    salvar_json(pasta / 'municipios_reservados.json', sorted(reservados))
    salvar_json(pasta / 'proveniencia.json', {
        'entrada_sha256': sha256(cfg['entrada']),
        'atributos_sha256': sha256(arquivo(cfg, 'ouro', 'atributos.parquet')),
        'schema': schema, 'cenario': 'retrospectivo; nao simula revisoes historicas',
    })
    resumo = {}
    total_ajustes = (len(cfg['treino']['horizontes']) * len(cfg['treino']['validacao_anos']) *
                     len(cfg['treino']['candidatos']))
    ajustes_concluidos = 0
    inicio_comparacao = time.perf_counter()
    LOG.info('Plano de treino: %d candidatos × %d horizontes × %d ano(s) = %d ajustes. Validação: %s.',
             len(cfg['treino']['candidatos']), len(cfg['treino']['horizontes']),
             len(cfg['treino']['validacao_anos']), total_ajustes, cfg['treino']['validacao_anos'])
    for h in cfg['treino']['horizontes']:
        LOG.info('Treinamento horizonte %d...', h)
        avaliacoes = []
        explicacoes = {}
        melhor_modelo, melhor_chave = None, None
        fold_unico = len(cfg['treino']['validacao_anos']) == 1
        for ano in cfg['treino']['validacao_anos']:
            fim, limite = pd.Timestamp(inicio_ano_epi(ano)), pd.Timestamp(inicio_ano_epi(ano+1))
            fit, cal = separar_ajuste_calibracao(cfg, dados[conhecidos], fim, h)
            avaliacao = dados[elegiveis(cfg, dados, h) & dados[f'data_alvo_h{h}'].ge(fim)
                             & dados[f'data_alvo_h{h}'].lt(limite)]
            if avaliacao.empty:
                raise ValueError(f'Sem linhas de validação em {ano}/h{h}')
            for tipo in cfg['treino']['candidatos']:
                LOG.info('[%d/%d | %.0f%%] INÍCIO h%d, validação %d, %s: %d treino + %d calibração + %d validação.',
                         ajustes_concluidos + 1, total_ajustes, 100 * ajustes_concluidos / total_ajustes,
                         h, ano, tipo, len(fit), len(cal), len(avaliacao))
                iniciado = time.perf_counter()
                try:
                    modelo = Candidato(cfg, tipo).fit(
                        fit, fit[f'alvo_h{h}'], schema['numericas'], schema['categoricas'],
                        data_alvo=f'data_alvo_h{h}')
                except (ValueError, FloatingPointError) as erro:
                    if tipo != 'nb_misto':
                        raise
                    LOG.warning('h%d, %d, nb_misto não convergiu: %s. Seguindo os demais modelos.', h, ano, erro)
                    avaliacoes.append({'ano': ano, 'grupo': 'conhecidos', 'modelo': tipo,
                                       'erro': str(erro), 'segundos_treino': round(time.perf_counter() - iniciado, 2)})
                    ajustes_concluidos += 1
                    LOG.info('[%d/%d | %.0f%%] FALHA %s/h%d após %s; comparação continua.',
                             ajustes_concluidos, total_ajustes, 100 * ajustes_concluidos / total_ajustes,
                             tipo, h, timedelta(seconds=int(time.perf_counter() - iniciado)))
                    continue
                explicacoes[tipo] = modelo.explicacao()
                limites = limiares(cfg, fit, fit[f'alvo_h{h}'])
                if cfg['treino'].get('progresso', {}).get('mostrar'):
                    LOG.info('%s/h%d: calibrando probabilidades e intervalos.', tipo, h)
                ajustar_calibracao(cfg, modelo, cal, cal[f'alvo_h{h}'])
                if tipo != 'poisson':
                    calibrar_limiar_ocorrencia(cfg, modelo, cal, cal[f'alvo_h{h}'])
                    if fold_unico:
                        cal_ocorrencia = metricas_ocorrencia(
                            cal[f'alvo_h{h}'], modelo.predict_ocorrencia(cal), modelo.limiar_ocorrencia,
                            float(cfg['treino']['ocorrencia'].get('casos_minimos', 1)))
                        cal_macro_mae = avaliar(cal, cal[f'alvo_h{h}'], modelo.predict(cal))['macro_mae']
                        piso_precision = float(cfg['treino']['ocorrencia'].get('precisao_minima', 0.80))
                        piso_recall = float(cfg['treino']['ocorrencia'].get('recall_minimo', 0.80))
                        chave = (
                            cal_ocorrencia['precision'] >= piso_precision and cal_ocorrencia['recall'] >= piso_recall,
                            cal_ocorrencia['f1'], cal_ocorrencia['pr_auc'] or 0.0, -cal_macro_mae,
                        )
                        if melhor_chave is None or chave > melhor_chave:
                            melhor_modelo, melhor_chave = modelo, chave
                metricas_conhecidos = None
                for grupo, mascara in [('conhecidos', ~avaliacao.codigo_ibge.isin(reservados)),
                                        ('nao_vistos', avaliacao.codigo_ibge.isin(reservados))]:
                    val = avaliacao[mascara]
                    if val.empty:
                        continue
                    threshold = aplicar_limiar(val, limites)
                    pred, low, high, prob = prever_com_intervalos(modelo, val, threshold)
                    m = avaliar(val, val[f'alvo_h{h}'], pred, low, high, threshold, prob)
                    if grupo == 'conhecidos':
                        metricas_conhecidos = m
                    if tipo != 'poisson':
                        m['ocorrencia'] = metricas_ocorrencia(
                            val[f'alvo_h{h}'], modelo.predict_ocorrencia(val), modelo.limiar_ocorrencia,
                            float(cfg['treino']['ocorrencia'].get('casos_minimos', 1)))
                    avaliacoes.append({'ano': ano, 'grupo': grupo, 'modelo': tipo,
                                       'segundos_treino': round(time.perf_counter() - iniciado, 2), 'metricas': m})
                ajustes_concluidos += 1
                decorrido = time.perf_counter() - inicio_comparacao
                eta = timedelta(seconds=int(decorrido / ajustes_concluidos *
                                            (total_ajustes - ajustes_concluidos)))
                ocorrencia = metricas_conhecidos.get('ocorrencia', {}) if metricas_conhecidos else {}
                LOG.info('[%d/%d | %.0f%%] FIM %s/h%d em %s | recall=%s precisão=%s PR-AUC=%s MAE macro=%s | ETA ~%s.',
                         ajustes_concluidos, total_ajustes, 100 * ajustes_concluidos / total_ajustes,
                         tipo, h, timedelta(seconds=int(time.perf_counter() - iniciado)),
                         f"{ocorrencia['recall']:.3f}" if 'recall' in ocorrencia else 'n/a',
                         f"{ocorrencia['precision']:.3f}" if 'precision' in ocorrencia else 'n/a',
                         f"{ocorrencia['pr_auc']:.3f}" if ocorrencia.get('pr_auc') is not None else 'n/a',
                         f"{metricas_conhecidos['macro_mae']:.3f}" if metricas_conhecidos else 'n/a', eta)
            for grupo, mascara in [('conhecidos', ~avaliacao.codigo_ibge.isin(reservados)),
                                    ('nao_vistos', avaliacao.codigo_ibge.isin(reservados))]:
                val = avaliacao[mascara]
                if val.empty:
                    continue
                for nome, serie in [('zero', pd.Series(0., index=val.index)),
                                    ('recente', val.baseline_recente), ('media4', val.baseline_media4),
                                    ('sazonal', val[f'baseline_sazonal_h{h}'])]:
                    ok = serie.notna()
                    if ok.any():
                        avaliacoes.append({'ano': ano, 'grupo': grupo, 'modelo': nome,
                                           'metricas': avaliar(val[ok], val.loc[ok, f'alvo_h{h}'], serie[ok])})
        salvar_json(pasta / f'validacao_h{h}.json', avaliacoes)
        salvar_json(pasta / f'explicacoes_h{h}.json', explicacoes)
        candidatos = cfg['treino']['candidatos']
        scores = {nome: float(np.mean([a['metricas']['macro_mae'] for a in avaliacoes
                                       if a['modelo'] == nome and a['grupo'] == 'conhecidos'
                                       and 'metricas' in a])) if any(a['modelo'] == nome and a['grupo'] == 'conhecidos'
                                                                    and 'metricas' in a for a in avaliacoes) else None
                  for nome in candidatos + ['zero', 'recente', 'media4', 'sazonal']}
        ocorrencia_scores = {nome: {
            chave: float(np.mean([a['metricas']['ocorrencia'][chave] for a in avaliacoes
                                   if a['modelo'] == nome and a['grupo'] == 'conhecidos'
                                   and 'ocorrencia' in a['metricas']]))
            for chave in ('recall', 'precision', 'f1', 'pr_auc')}
            for nome in candidatos if nome != 'poisson' and any(a['modelo'] == nome and
                a['grupo'] == 'conhecidos' and 'ocorrencia' in a.get('metricas', {}) for a in avaliacoes)}
        if fold_unico and melhor_modelo is not None:
            vencedor = melhor_modelo.tipo
        else:
            vencedor = min((n for n in candidatos if scores[n] is not None), key=scores.get)
        fim_final = pd.Timestamp(inicio_ano_epi(cfg['treino']['treino_final_ate'] + 1))
        if fold_unico and pd.Timestamp(inicio_ano_epi(cfg['treino']['validacao_anos'][0])) == fim_final and melhor_modelo is not None:
            modelo = melhor_modelo
            fit, cal = separar_ajuste_calibracao(cfg, dados[conhecidos], fim_final, h)
            modelo.limiares = limiares(cfg, fit, fit[f'alvo_h{h}'])
        else:
            fit, cal = separar_ajuste_calibracao(cfg, dados[conhecidos], fim_final, h)
            modelo = Candidato(cfg, vencedor).fit(
                fit, fit[f'alvo_h{h}'], schema['numericas'], schema['categoricas'],
                data_alvo=f'data_alvo_h{h}')
            modelo.limiares = limiares(cfg, fit, fit[f'alvo_h{h}'])
            ajustar_calibracao(cfg, modelo, cal, cal[f'alvo_h{h}'])
            if vencedor != 'poisson':
                calibrar_limiar_ocorrencia(cfg, modelo, cal, cal[f'alvo_h{h}'])
        modelo.horizonte = h
        modelo.schema = schema
        modelo.data_fim_treino = str(fit[f'data_alvo_h{h}'].max().date())
        modelo.data_fim_calibracao = str(cal[f'data_alvo_h{h}'].max().date())
        joblib.dump(modelo, pasta / f'modelo_h{h}.joblib', compress=3)
        salvar_json(pasta / f'imputacao_h{h}.json', modelo.imputador.relatorio)
        salvar_json(pasta / f'smote_h{h}.json', modelo.smote)
        resumo[str(h)] = {'candidato': vencedor, 'scores_macro_mae': scores,
                          'scores_ocorrencia': ocorrencia_scores,
                          'criterio_vencedor': 'recall_ocorrencia_com_piso_precisao' if melhor_modelo is not None else 'mae_macro',
                          'melhor_baseline': min(['zero', 'recente', 'media4', 'sazonal'], key=scores.get),
                          'supera_baselines': scores[vencedor] < min(scores[n] for n in ['zero', 'recente', 'media4', 'sazonal']),
                          'aprovado_ocorrencia': (vencedor in ocorrencia_scores and
                                                  ocorrencia_scores[vencedor]['precision'] >= cfg['treino']['ocorrencia']['precisao_minima']),
                          'treino_ate': modelo.data_fim_treino, 'calibracao_ate': modelo.data_fim_calibracao,
                          'features': len(modelo.imputador.ativas), 'smote_sinteticos_final': modelo.smote.get('sinteticos', 0),
                          'limiar_ocorrencia': getattr(modelo, 'limiar_ocorrencia', None),
                          'calibracao_ocorrencia': getattr(modelo, 'calibracao_ocorrencia', None)}
        LOG.info('Horizonte h%d concluído: vencedor %s; modelo e relatórios em %s.', h, vencedor, pasta)
    salvar_json(pasta / 'resumo.json', resumo)
    salvar_json(pasta / 'model_card.json', {
        'status': 'candidato_de_pesquisa; aprovacao epidemiologica pendente',
        'testes_bloqueados_abertos': False,
        'teste_independente_disponivel': bool(cfg['treino']['testes_anos']),
        'anos_validacao': cfg['treino']['validacao_anos'],
        'limitacoes': ['snapshot revisado sem vintages historicos', 'zeros inferidos do recurso anual',
                      'malha fixa de referencia', 'clima de reanalise nao previsao meteorologica',
                      'intervalos residuais sem garantia sob dependencia temporal',
                      'limiar de alerta experimental, nao norma sanitaria'],
        'resumo': resumo,
    })
    salvar_json(arquivo(cfg, 'modelos', 'ultimo_treino.json'), {'pasta': str(pasta)})
    LOG.info('Treino concluído: %s', pasta)
    return pasta


In [ ]:
def arquivar_candidatos(cfg, pasta=None):
    """Preserva horizontes promissores em contagem ou ocorrência de casos."""
    # Arquivar é leitura de um experimento já concluído: não imponha a
    # configuração corrente, pois ela pode ter sido ajustada para o próximo treino.
    if pasta is None:
        pasta = json.loads(arquivo(cfg, 'modelos', 'ultimo_treino.json').read_text())['pasta']
    pasta = Path(pasta)
    if not (pasta / 'resumo.json').is_file():
        raise FileNotFoundError(f'Não encontrei resumo.json em {pasta}')
    resumo = json.loads((pasta / 'resumo.json').read_text())
    horizontes = [int(h) for h, meta in resumo.items()
                  if meta.get('supera_baselines') or meta.get('aprovado_ocorrencia')]
    if not horizontes:
        raise ValueError('Nenhum candidato atingiu os critérios de contagem ou ocorrência na validação.')
    destino_base = Path(cfg['treino']['arquivo_candidatos'])
    if not destino_base.is_absolute():
        destino_base = Path.cwd() / destino_base
    destino = destino_base / pasta.name
    destino.mkdir(parents=True, exist_ok=True)
    comuns = ['config.json', 'ambiente.json', 'proveniencia.json', 'model_card.json',
              'municipios_reservados.json', 'resumo.json', 'testes_bloqueados.json']
    selecionados = comuns + sum(([f'modelo_h{h}.joblib', f'imputacao_h{h}.json', f'smote_h{h}.json',
                                   f'validacao_h{h}.json', f'explicacoes_h{h}.json'] for h in horizontes), [])
    for nome in selecionados:
        origem = pasta / nome
        if origem.is_file():
            shutil.copy2(origem, destino / nome)
    salvar_json(destino / 'selecao.json', {
        'origem': str(pasta), 'horizontes_preservados': horizontes,
        'regra': 'venceu baselines de contagem ou atingiu piso de precisao para ocorrencia na validacao',
        'data_arquivo': datetime.now(timezone.utc).isoformat(),
        'nao_e_aprovacao_producao': True,
    })
    LOG.info('Candidatos arquivados: %s', destino)
    return destino


def localizar_modelos(cfg, pasta=None):
    if pasta is None:
        pasta = json.loads(arquivo(cfg, 'modelos', 'ultimo_treino.json').read_text())['pasta']
    from pathlib import Path
    pasta = Path(pasta)
    origem = json.loads((pasta / 'config.json').read_text())
    if origem != cfg:
        raise ValueError('Configuração mudou desde o treino. Use o config.yaml original ou treine novamente.')
    return pasta


def avaliar_testes(cfg, pasta=None):
    """Comando explícito abre 2024/2025; jamais usado para selecionar o vencedor."""
    if not cfg['treino']['testes_anos']:
        raise ValueError('Não há ano de teste independente: 2025 foi usado para validação, não existe teste independente configurado.')
    dados, _, reservados = carregar_dados(cfg)
    pasta = localizar_modelos(cfg, pasta)
    report = []
    for h in cfg['treino']['horizontes']:
        modelo = joblib.load(pasta / f'modelo_h{h}.joblib')
        for ano in cfg['treino']['testes_anos']:
            first, last = pd.Timestamp(inicio_ano_epi(ano)), pd.Timestamp(inicio_ano_epi(ano+1))
            base = dados[elegiveis(cfg, dados, h) & dados[f'data_alvo_h{h}'].ge(first) & dados[f'data_alvo_h{h}'].lt(last)]
            for grupo, mask in [('conhecidos', ~base.codigo_ibge.isin(reservados)),
                                 ('nao_vistos', base.codigo_ibge.isin(reservados))]:
                df = base[mask]
                if df.empty:
                    continue
                threshold = aplicar_limiar(df, modelo.limiares)
                pred, lo, hi, prob = prever_com_intervalos(modelo, df, threshold)
                metricas_teste = avaliar(df, df[f'alvo_h{h}'], pred, lo, hi, threshold, prob)
                if modelo.tipo in ('hurdle', 'arima'):
                    metricas_teste['ocorrencia'] = metricas_ocorrencia(
                        df[f'alvo_h{h}'], modelo.predict_ocorrencia(df),
                        getattr(modelo, 'limiar_ocorrencia', 0.5),
                        float(cfg['treino']['ocorrencia'].get('casos_minimos', 1)))
                report.append({'ano': ano, 'h': h, 'grupo': grupo, 'modelo': modelo.tipo,
                               'metricas': metricas_teste})
                output = df[['codigo_ibge', 'uf', 'porte', 'semana_inicio', f'data_alvo_h{h}', f'alvo_h{h}']].copy()
                output['previsao'], output['inferior'], output['superior'], output['prob_alerta'] = pred, lo, hi, prob
                salvar_parquet(output, pasta / f'previsoes_teste_{ano}_{grupo}_h{h}.parquet')
                for nome in ['recente', 'media4', f'sazonal_h{h}']:
                    serie = df['baseline_' + nome]
                    ok = serie.notna()
                    if ok.any():
                        report.append({'ano': ano, 'h': h, 'grupo': grupo, 'modelo': 'baseline_' + nome,
                                       'metricas': avaliar(df[ok], df.loc[ok, f'alvo_h{h}'], serie[ok])})
    salvar_json(pasta / 'testes_bloqueados.json', report)
    card = json.loads((pasta / 'model_card.json').read_text())
    card['testes_bloqueados_abertos'] = True
    card['testes_abertos_em'] = datetime.now(timezone.utc).isoformat()
    salvar_json(pasta / 'model_card.json', card)
    LOG.info('Testes avaliados e registrados: %s', pasta / 'testes_bloqueados.json')


def inferir(cfg, pasta=None):
    dados, _, _ = carregar_dados(cfg)
    pasta = localizar_modelos(cfg, pasta)
    ultima = dados.semana_inicio.max()
    base = dados[dados.semana_inicio.eq(ultima)].copy()
    delay = cfg['atributos']['atraso_casos_semanas']
    antigo = delay + max(cfg['atributos']['lags'])
    base = base[base[f'f_casos_lag{delay}'].notna() & base[f'f_casos_lag{antigo}'].notna()]
    if base.empty:
        raise ValueError('Sem histórico elegível para inferência na última origem.')
    rows = []
    for h in cfg['treino']['horizontes']:
        modelo = joblib.load(pasta / f'modelo_h{h}.joblib')
        threshold = aplicar_limiar(base, modelo.limiares)
        pred, low, high, prob = prever_com_intervalos(modelo, base, threshold)
        out = base[['codigo_ibge', 'uf', 'semana_inicio', 'populacao']].copy()
        out['semana_alvo'] = out.semana_inicio + pd.Timedelta(weeks=h)
        out['horizonte'], out['casos_previstos'], out['incidencia_100k'] = h, pred, pred/out.populacao*100000
        out['limite_inferior'], out['limite_superior'], out['prob_alerta_experimental'] = low, high, prob
        if modelo.tipo in ('hurdle', 'arima'):
            out['prob_ocorrencia'] = modelo.predict_ocorrencia(base)
            out['limiar_ocorrencia'] = getattr(modelo, 'limiar_ocorrencia', 0.5)
            out['ocorrencia_prevista'] = out['prob_ocorrencia'] >= out['limiar_ocorrencia']
        out['limiar_experimental'], out['model_version'] = threshold, pasta.name
        out['status'] = 'pesquisa_retrospectiva_snapshot_revisado'
        rows.append(out)
    output = pd.concat(rows, ignore_index=True)
    salvar_parquet(output, pasta / 'previsoes_ultima_origem.parquet')
    LOG.info('Previsões da origem %s: %s', ultima.date(), pasta / 'previsoes_ultima_origem.parquet')
    return output


In [ ]:
TIPOS_ADICIONAIS = TIPOS


## 4. Dataset SINAN e configuração de Pernambuco

Envie o Parquet completo ou todas as partes sequenciais para `/content` (ou `/content/pacoteColab`). Se os dados ainda não estiverem no runtime, a célula abre a janela de upload.


In [ ]:
import shutil
from pathlib import Path
import duckdb

pastas_dados = [Path('/content'), Path('/content/pacoteColab'), Path('.'), Path('./pacoteColab')]


def localizar_entradas():
    completos, grupos = [], []
    for pasta in pastas_dados:
        if not pasta.is_dir():
            continue
        completo = pasta / 'dengue-analitico.parquet'
        if completo.is_file() and completo not in completos:
            completos.append(completo)
        partes = sorted(pasta.glob('dengue-analitico.parquet.part[0-9][0-9][0-9]'))
        if partes:
            grupos.append(partes)
    return completos, grupos


parquets, grupos_partes = localizar_entradas()
if len(parquets) > 1:
    raise FileExistsError(f'Há mais de um Parquet completo: {parquets}. Deixe somente o dataset escolhido.')
if parquets and grupos_partes:
    raise FileExistsError('Foram encontrados um Parquet completo e partes. Deixe somente uma forma de entrada.')
if parquets:
    DATASET = parquets[0]
else:
    if len(grupos_partes) != 1:
        raise FileNotFoundError(
            'Coloque dengue-analitico.parquet ou todas as partes sequenciais em '
            '/content, /content/pacoteColab, . ou ./pacoteColab.'
        )
    partes = grupos_partes[0]
    indices = [int(parte.name.rsplit('part', 1)[1]) for parte in partes]
    if indices != list(range(1, len(partes) + 1)):
        raise ValueError(f'Partes ausentes ou fora de ordem: {indices}')
    DATASET = partes[0].parent / 'dengue-analitico-reconstruido.parquet'
    temporario = DATASET.with_name(DATASET.name + '.partial')
    with temporario.open('wb') as saida:
        for parte in partes:
            with parte.open('rb') as entrada:
                shutil.copyfileobj(entrada, saida, length=1024 * 1024)
    temporario.replace(DATASET)

con = duckdb.connect()
try:
    colunas = {linha[0] for linha in con.execute(
        'DESCRIBE SELECT * FROM read_parquet(?)', [str(DATASET)]
    ).fetchall()}
finally:
    con.close()
obrigatorias = {'ID_MN_RESI', 'SEM_PRI', 'DT_SIN_PRI', 'CLASSI_FIN', 'ANO_ARQUIVO'}
if obrigatorias - colunas:
    raise ValueError(f'Dataset SINAN incompatível; faltam colunas: {sorted(obrigatorias - colunas)}')
print(f'Dataset pronto: {DATASET} ({DATASET.stat().st_size / 2**30:.2f} GiB)')

## 5. Configuração embutida

A configuração original está incorporada nesta célula. O notebook altera somente entrada, diretório de saída e recorte para Pernambuco.


In [ ]:
CONFIG = yaml.safe_load(r'''
# Caminhos relativos à raiz do repositório. Python 3.10–3.12 recomendado.
entrada: data/sinan/dengue-analitico.parquet
saida: treinamentoDoModelo/artefatos
ano_inicio: 2015
ano_fim: 2025
data_corte: '2026-11-05'
ufs: ['21', '22', '23', '24', '25', '26', '27', '28', '29']
municipios: []  # vazio = todos; teste rápido: ['2611606', '2304400', '2927408']
threads: 4
memoria_duckdb: 2GB
semente: 42
offline: false  # true: só lê os downloads explicitamente preparados nesta saída

painel:
  classificacoes: ['10', '11', '12']
  id_alvo: confirmados_classi_fin_10_11_12_inicio_sintomas_v1
  maturacao_dias: 90
  # O SINAN aberto não comprova notificação negativa por município/semana.
  # Os zeros do snapshot completo são INFERIDOS, com flag no painel e relatório.
  zeros: snapshot_anual
  cobertura_csv: null  # código, semana_inicio, fonte_recebida (0/1)
  mapa_municipios_csv: null  # substitui dimensão atual por vínculos com vigência
  # Não remover coincidências demográficas: podem representar pessoas diferentes.
  excluir_duplicidade_marcada: true  # somente NDUPLIC_N=2, contabilizado no relatório
  max_fracao_quarentena: 0.05

geografia:
  periodo: 2022
  qualidade: intermediaria
  # Malha fixa de referência; não é uma reconstrução histórica de fronteiras.

populacao:
  defasagem_publicacao_dias: 366
  censo2022_disponivel_em: '2023-06-28'
  max_idade_anos: 3
  # Âncoras oficiais anteriores ao início permitem ffill causal, nunca bfill.
  ano_ancora_inicio: 2013

clima:
  provedor: nasa_power  # nasa_power | inmet | nenhum (ablação explícita)
  workers: 3
  lat_grade: 0.5
  lon_grade: 0.625
  min_dias: 6
  atraso_semanas: 1
  ffill_semanas: 2
  max_fracao_ausente: 0.60
  inmet_zips: []  # vazio: descobre os ZIPs anuais no portal oficial
  inmet_workers: 2
  inmet_raio_km: 150
  inmet_estacoes: 3
  inmet_min_cobertura: 0.80

# Tabelas agregadas opcionais com vigência e disponibilidade. Nunca usa estoque
# atual como histórico. Veja ENRIQUECIMENTO.md para o contrato exato.
fontes_opcionais: []
# - nome: cnes
#   caminho: dados/cnes_municipio.csv
#   colunas: [qtd_cnes]
#   max_idade_dias: 93
# - nome: liraa
#   caminho: dados/liraa_municipio.csv
#   colunas: [indice_liraa]
#   max_idade_dias: 120

atributos:
  # Usa história final com margem conservadora. Sem snapshots históricos,
  # o experimento continua retrospectivo, não backtest operacional as-of.
  atraso_casos_semanas: 2
  lags: [0, 1, 2, 3, 4, 8, 13, 26, 52]
  janelas: [2, 4, 8, 12, 13, 26]
  clima_lags: [0, 1, 2, 4, 8, 12]
  clima_janelas: [2, 4, 8, 12]

treino:
  horizontes: [1, 2]
  # Comparação temporal exclusiva em 2025; treino e calibração terminam em 2024.
  # Não há ano posterior reservado para teste independente neste recorte.
  validacao_anos: [2025]
  testes_anos: []
  treino_final_ate: 2024
  fracao_municipios_reservados: 0.10
  candidatos: [poisson, hurdle, logistica_poisson, ebm, xgboost, lightgbm, random_forest, extra_trees, bayes_hierarquico, nb_misto, arima]
  max_linhas_modelos_adicionais: 120000
  max_atributos_ebm: 30
  arima:
    ordem: [1, 1, 1]
    min_semanas: 52
  treinamento_rapido: true
  progresso:
    mostrar: false  # notebook Colab ativa; CatBoost imprime a cada intervalo_arvores
    intervalo_arvores: 50
  iterations: 350
  depth: 5
  learning_rate: 0.05
  l2_leaf_reg: 5
  early_stopping_rounds: 60
  redes_neurais:
    seq_len: 13
    batch_size: 512
    epocas: 35
    paciencia: 7
    taxa_aprendizado: 0.001
    peso_regressao: 0.35
  calibracao_semanas: 26
  cobertura_intervalo: 0.90
  # Limite EXPERIMENTAL, não definição oficial de epidemia.
  alerta_quantil: 0.95
  alerta_min_casos: 5
  arquivo_candidatos: treinamentoDoModelo/modelosCandidatos
  # Operação de alta sensibilidade: o corte é escolhido na calibração para
  # maximizar recall, desde que a precisão não caia abaixo deste piso.
  ocorrencia:
    casos_minimos: 1  # 1 = casos > 0; use 3 para surto relevante >= 3
    precisao_minima: 0.80
    recall_minimo: 0.80
    limiar_padrao: 0.50
  smote:
    habilitado: true
    # Razão máxima positivos:zeros após SMOTE, aplicada separadamente por UF.
    # O painel parte de cerca de 16 positivos para cada 100 semanas zero.
    razao_pos_neg_max: 0.20
    k_vizinhos: 2
    max_sinteticos_por_uf: 50000
  # Busca opcional separada; em modo de validação única usa somente 2025.
  busca_hiperparametros:
    habilitado: false
    anos_triagem: [2025]
    anos_confirmacao: [2025]
    horizontes_triagem: [1]
    horizontes_confirmacao: [1, 2]
    finalistas: 2
    tentativas:
      - nome: referencia_smote
        iterations: 300
        depth: 5
        learning_rate: 0.05
        l2_leaf_reg: 5
        peso_classe_positiva: 1.0
        smote: {habilitado: true, razao_pos_neg_max: 0.20}
      - nome: peso_positivo_4
        iterations: 260
        depth: 5
        learning_rate: 0.06
        l2_leaf_reg: 8
        peso_classe_positiva: 4.0
        smote: {habilitado: false}
      - nome: peso_positivo_6
        iterations: 260
        depth: 5
        learning_rate: 0.06
        l2_leaf_reg: 8
        peso_classe_positiva: 6.0
        smote: {habilitado: false}
      - nome: peso_positivo_8
        iterations: 250
        depth: 5
        learning_rate: 0.06
        l2_leaf_reg: 10
        peso_classe_positiva: 8.0
        smote: {habilitado: false}
      - nome: peso_positivo_10
        iterations: 250
        depth: 4
        learning_rate: 0.07
        l2_leaf_reg: 10
        peso_classe_positiva: 10.0
        smote: {habilitado: false}
''')
CONFIG['entrada'] = str(DATASET.resolve())
CONFIG['saida'] = str(ROOT / 'treinamentoDoModelo' / 'artefatos')
CONFIG['ufs'] = ['26']
CONFIG['municipios'] = []
CONFIG['offline'] = False
CONFIG['clima']['provedor'] = 'nenhum'
CONFIG['treino']['progresso']['mostrar'] = True
CONFIG['treino']['progresso']['intervalo_arvores'] = 50
CONFIG['treino']['candidatos'] = [
    nome for nome in CONFIG['treino']['candidatos']
    if nome not in ('random_forest', 'extra_trees')
]
CONFIG = carregar_config(CONFIG)
print('UF:', CONFIG['ufs'], '| entrada:', CONFIG['entrada'])
print('Modelos:', ', '.join(CONFIG['treino']['candidatos']))


## 6. Preparar e treinar

A execução abaixo chama diretamente as funções do pipeline carregadas pelas células anteriores: prepara fontes, painel e atributos e, em seguida, treina e valida os candidatos.


In [ ]:
# Correção temporária para tratar inconsistências de digitação no SINAN para PE (Ex: Quipapá 2611533 -> 2611531)
_original_codigo_ibge_completo = codigo_ibge_completo

def codigo_ibge_completo_corrigido(codigo):
    codigo_str = str(codigo).strip()
    # Corrige explicitamente o dígito verificador incorreto gerado na entrada de dados
    if codigo_str == '2611533':
        return '2611531'
    return _original_codigo_ibge_completo(codigo_str)

# Sobrescreve globalmente a função de validação para tolerar esse desvio estrutural do SINAN
codigo_ibge_completo = codigo_ibge_completo_corrigido

inicio_execucao = datetime.now(timezone.utc)
preparar_fontes(CONFIG)
construir_painel(CONFIG)
construir_atributos(CONFIG)

pasta_treino = treinar(CONFIG)
duracao_execucao = datetime.now(timezone.utc) - inicio_execucao
print(f'Treinamento concluído em {duracao_execucao}.')
print(f'Resultados: {pasta_treino}')

## 7. Resultados e download


In [ ]:
from IPython.display import display

registros = []
for relatorio in sorted(pasta_treino.glob('validacao_h*.json')):
    for item in json.loads(relatorio.read_text(encoding='utf-8')):
        if item.get('grupo') != 'conhecidos':
            continue
        metricas = item.get('metricas', {})
        ocorrencia = metricas.get('ocorrencia', {})
        registros.append({
            'horizonte': relatorio.stem.replace('validacao_', ''),
            'modelo': item.get('modelo'),
            'recall': ocorrencia.get('recall'),
            'precisao': ocorrencia.get('precision'),
            'PR-AUC': ocorrencia.get('pr_auc'),
            'MAE macro': metricas.get('macro_mae'),
            'segundos': item.get('segundos_treino'),
            'erro': item.get('erro'),
        })
if registros:
    display(pd.DataFrame(registros).sort_values(
        ['horizonte', 'MAE macro'], na_position='last'))

resumo_path = pasta_treino / 'resumo.json'
if resumo_path.is_file():
    resumo = json.loads(resumo_path.read_text(encoding='utf-8'))
    print('Vencedores:', json.dumps(
        {h: registro.get('candidato') for h, registro in resumo.items()},
        ensure_ascii=False,
    ))

arquivo_saida = Path('/content/resultados_treinamento_pe.zip')
with zipfile.ZipFile(arquivo_saida, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as pacote:
    for camada in ('modelos', 'relatorios'):
        pasta_camada = Path(CONFIG['saida']) / camada
        for arquivo in pasta_camada.rglob('*'):
            if arquivo.is_file():
                pacote.write(arquivo, arcname=str(arquivo.relative_to(CONFIG['saida'])))
print(f'Resultados prontos: {arquivo_saida} ({arquivo_saida.stat().st_size / 2**20:.1f} MiB)')
files.download(str(arquivo_saida))


In [ ]:
import json
import numpy as np
import pandas as pd

caminho_atributos = arquivo(CONFIG, 'ouro', 'atributos.parquet')
dados = pd.read_parquet(caminho_atributos)
dados['codigo_ibge'] = dados.codigo_ibge.astype(str)
dados['uf'] = dados.uf.astype(str)
dados = dados.sort_values(['codigo_ibge', 'semana_inicio']).reset_index(drop=True)
schema = json.loads(arquivo(CONFIG, 'ouro', 'schema_atributos.json').read_text())
reservados_path = arquivo(CONFIG, 'ouro', 'municipios_reservados.json')
reservados = set(json.loads(reservados_path.read_text())) if reservados_path.is_file() else set()
conhecidos = ~dados.codigo_ibge.isin(reservados)
features_input = [c for c in schema['numericas'] if c in dados.columns and c.startswith('f_')]

print(
    f"Painel neural: {len(dados):,} linhas | {len(features_input)} atributos | "
    f"municípios reservados: {len(reservados)}"
)


## 8. Suíte temporal de redes neurais e ensembles

Sete arquiteturas usam janelas de 13 semanas, cabeças independentes para contagem e probabilidade, imputação/normalização ajustadas no treino e limiares calibrados antes de 2025.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def criar_cabecas(dimensao):
    head_count = nn.Sequential(nn.Linear(dimensao, 48), nn.GELU(), nn.Dropout(0.10), nn.Linear(48, 1))
    head_prob = nn.Sequential(nn.Linear(dimensao, 48), nn.LayerNorm(48), nn.GELU(), nn.Dropout(0.10), nn.Linear(48, 1))
    return head_count, head_prob


class DengueLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.15)
        self.head_count, self.head_prob = criar_cabecas(hidden_dim)

    def forward(self, x):
        encoded, _ = self.encoder(x)
        last = encoded[:, -1]
        return self.head_count(last).squeeze(-1), self.head_prob(last).squeeze(-1)


class DengueGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.encoder = nn.GRU(input_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.15)
        self.head_count, self.head_prob = criar_cabecas(hidden_dim)

    def forward(self, x):
        encoded, _ = self.encoder(x)
        last = encoded[:, -1]
        return self.head_count(last).squeeze(-1), self.head_prob(last).squeeze(-1)


class DengueBiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=40):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.15)
        self.head_count, self.head_prob = criar_cabecas(hidden_dim * 2)

    def forward(self, x):
        encoded, _ = self.encoder(x)
        last = encoded[:, -1]
        return self.head_count(last).squeeze(-1), self.head_prob(last).squeeze(-1)


class DengueCNN1D(nn.Module):
    def __init__(self, input_dim, channels=48):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(channels, channels * 2, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head_count, self.head_prob = criar_cabecas(channels * 2)

    def forward(self, x):
        x = F.gelu(self.conv1(x.transpose(1, 2)))
        x = F.gelu(self.conv2(x))
        pooled = self.pool(x).squeeze(-1)
        return self.head_count(pooled).squeeze(-1), self.head_prob(pooled).squeeze(-1)


class DengueGRU_CNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=48):
        super().__init__()
        self.conv = nn.Conv1d(input_dim, 48, kernel_size=3, padding=1)
        self.encoder = nn.GRU(48, hidden_dim, batch_first=True)
        self.head_count, self.head_prob = criar_cabecas(hidden_dim)

    def forward(self, x):
        convolved = F.gelu(self.conv(x.transpose(1, 2))).transpose(1, 2)
        encoded, _ = self.encoder(convolved)
        last = encoded[:, -1]
        return self.head_count(last).squeeze(-1), self.head_prob(last).squeeze(-1)


class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation):
        super().__init__()
        self.dilation = dilation
        self.conv1 = nn.Conv1d(in_channels, out_channels, 3, padding=2 * dilation, dilation=dilation)
        self.conv2 = nn.Conv1d(out_channels, out_channels, 3, padding=2 * dilation, dilation=dilation)
        self.skip = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        self.dropout = nn.Dropout(0.10)

    def forward(self, x):
        first = self.conv1(x)[..., :-2 * self.dilation]
        first = self.dropout(F.gelu(first))
        second = self.conv2(first)[..., :-2 * self.dilation]
        return F.gelu(self.skip(x) + self.dropout(second))


class DengueTCN(nn.Module):
    def __init__(self, input_dim, channels=48):
        super().__init__()
        self.block1 = TCNBlock(input_dim, channels, dilation=1)
        self.block2 = TCNBlock(channels, channels, dilation=2)
        self.head_count, self.head_prob = criar_cabecas(channels)

    def forward(self, x):
        encoded = self.block2(self.block1(x.transpose(1, 2)))
        last = encoded[:, :, -1]
        return self.head_count(last).squeeze(-1), self.head_prob(last).squeeze(-1)


class DengueResMLP(nn.Module):
    def __init__(self, input_dim, seq_len=13, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(input_dim * seq_len, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.head_count, self.head_prob = criar_cabecas(hidden_dim)

    def forward(self, x):
        flat = x.reshape(x.shape[0], -1)
        hidden = F.gelu(self.norm1(self.fc1(flat)))
        hidden = F.gelu(self.norm2(self.fc2(hidden) + hidden))
        return self.head_count(hidden).squeeze(-1), self.head_prob(hidden).squeeze(-1)


ARQUITETURAS = {
    'DengueLSTM': DengueLSTM,
    'DengueGRU': DengueGRU,
    'DengueBiLSTM': DengueBiLSTM,
    'DengueCNN1D': DengueCNN1D,
    'DengueGRU_CNN': DengueGRU_CNN,
    'DengueTCN': DengueTCN,
    'DengueResMLP': DengueResMLP,
}


### Treino, calibração e comparação

A validação de 2025 é usada somente para avaliação; a calibração, a seleção do Top-4 e os limiares ocorrem no bloco temporal anterior.

In [ ]:
import copy
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, precision_recall_curve, precision_recall_fscore_support


NEURAL_CFG = CONFIG['treino'].get('redes_neurais', {})
SEQ_LEN = int(NEURAL_CFG.get('seq_len', 13))
BATCH_SIZE = int(NEURAL_CFG.get('batch_size', 512))
EPOCHS = int(NEURAL_CFG.get('epocas', 35))
PATIENCE = int(NEURAL_CFG.get('paciencia', 7))
CASOS_MINIMOS = float(CONFIG['treino']['ocorrencia'].get('casos_minimos', 1))
PRECISAO_MINIMA = float(CONFIG['treino']['ocorrencia'].get('precisao_minima', 0.80))
RECALL_MINIMO = float(CONFIG['treino']['ocorrencia'].get('recall_minimo', 0.80))
PESO_REGRESSAO = float(NEURAL_CFG.get('peso_regressao', 0.35))

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
use_cuda_amp = device.type == 'cuda'
print(f'Accelerator: {device} | janela: {SEQ_LEN} semanas | batch: {BATCH_SIZE}')


def mascara_elegivel(df, horizonte):
    antigo = CONFIG['atributos']['atraso_casos_semanas'] + max(CONFIG['atributos']['lags'])
    return (
        df[f'alvo_h{horizonte}'].notna()
        & df[f'f_casos_lag{antigo}'].notna()
        & df.baseline_recente.notna()
        & df.baseline_media4.notna()
        & df[f'baseline_sazonal_h{horizonte}'].notna()
    )


def preparar_tensores(df, indices, horizonte, colunas, indices_treino, seq_len, preprocessor=None):
    """Imputa pela mediana e normaliza somente com estatísticas do treino."""
    if preprocessor is None:
        bruto = df[colunas].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
        ajuste = bruto.iloc[indices_treino]
        medianas = ajuste.median()
        ativas = medianas.index[medianas.notna()].tolist()
        if not ativas:
            raise ValueError('Nenhuma feature possui observações no conjunto de treino.')
        medianas = medianas[ativas]
        ajuste = ajuste[ativas].fillna(medianas)
        medias = ajuste.mean()
        desvios = ajuste.std(ddof=0).replace(0, 1.0).fillna(1.0)
        valores = bruto[ativas].fillna(medianas)
        matriz = ((valores - medias) / desvios).to_numpy(dtype=np.float32)
        if not np.isfinite(matriz).all():
            raise ValueError('Persistem NaN/Inf após imputação mediana do treino.')
        preprocessor = {
            'features': ativas, 'medianas': medianas, 'medias': medias,
            'desvios': desvios, 'matriz': matriz,
        }

    wanted = np.zeros(len(df), dtype=bool)
    wanted[np.asarray(indices, dtype=int)] = True
    datas = pd.to_datetime(df.semana_inicio).to_numpy(dtype='datetime64[D]')
    x_parts, ref_parts = [], []
    matriz = preprocessor['matriz']
    for posicoes in df.groupby('codigo_ibge', sort=False).indices.values():
        posicoes = np.asarray(posicoes, dtype=int)
        if len(posicoes) < seq_len:
            continue
        janelas = np.lib.stride_tricks.sliding_window_view(
            matriz[posicoes], window_shape=seq_len, axis=0
        ).transpose(0, 2, 1)
        janelas_data = np.lib.stride_tricks.sliding_window_view(datas[posicoes], window_shape=seq_len)
        finais = posicoes[seq_len - 1:]
        mascara = wanted[finais]
        mascara &= np.all(np.diff(janelas_data, axis=1) == np.timedelta64(7, 'D'), axis=1)
        if mascara.any():
            x_parts.append(janelas[mascara])
            ref_parts.append(finais[mascara])

    if not ref_parts:
        raise ValueError(f'Sem janelas completas de {seq_len} semanas para h{horizonte}.')
    referencias = np.concatenate(ref_parts)
    x = np.concatenate(x_parts).astype(np.float32, copy=False)
    referencia = df.iloc[referencias].copy().reset_index(drop=True)
    y = pd.to_numeric(referencia[f'alvo_h{horizonte}'], errors='raise').to_numpy(dtype=np.float32)
    if not np.isfinite(y).all() or (y < 0).any():
        raise ValueError('Alvos de treino precisam ser finitos e não negativos.')
    return torch.from_numpy(x), torch.from_numpy(y), referencia, preprocessor


def escolher_limiar(y, scores):
    y_bin = (np.asarray(y, dtype=float) >= CASOS_MINIMOS).astype(int)
    scores = np.asarray(scores, dtype=float)
    if not y_bin.any() or y_bin.all():
        return 0.5, False
    precisions, recalls, thresholds = precision_recall_curve(y_bin, scores)
    if not len(thresholds):
        return 0.5, False
    f1 = 2 * precisions[:-1] * recalls[:-1] / np.maximum(precisions[:-1] + recalls[:-1], 1e-12)
    validos = np.flatnonzero((precisions[:-1] >= PRECISAO_MINIMA) & (recalls[:-1] >= RECALL_MINIMO))
    if len(validos):
        return float(thresholds[validos[np.argmax(f1[validos])]]), True
    candidatos = np.flatnonzero(precisions[:-1] >= PRECISAO_MINIMA)
    if len(candidatos):
        return float(thresholds[candidatos[np.argmax(f1[candidatos])]]), False
    return float(thresholds[np.argmax(f1)]), False


def medir_modelo(y, contagens, scores, referencias, limiar):
    y = np.asarray(y, dtype=float)
    contagens = np.maximum(np.asarray(contagens, dtype=float), 0)
    scores = np.asarray(scores, dtype=float)
    alvo_binario = (y >= CASOS_MINIMOS).astype(int)
    previsto = scores >= limiar
    precisao, recall, f1, _ = precision_recall_fscore_support(
        alvo_binario, previsto, average='binary', zero_division=0
    )
    pr_auc = average_precision_score(alvo_binario, scores) if alvo_binario.any() else np.nan
    erros = pd.DataFrame({
        'codigo_ibge': referencias.codigo_ibge.to_numpy(),
        'erro': np.abs(y - contagens),
    })
    mae_macro = float(erros.groupby('codigo_ibge', observed=True).erro.mean().mean())
    return {
        'PR-AUC': float(pr_auc), 'F1': float(f1), 'Recall': float(recall),
        'Precisão': float(precisao), 'MAE macro': mae_macro,
        'Limiar': float(limiar),
        'Meta P/R >= 0.80': bool(precisao >= PRECISAO_MINIMA and recall >= RECALL_MINIMO),
    }


def inferir_lotes(modelo, x, batch_size=4096):
    contagens, scores = [], []
    modelo.eval()
    with torch.inference_mode():
        for inicio in range(0, len(x), batch_size):
            lote = x[inicio:inicio + batch_size]
            if use_cuda_amp:
                with torch.amp.autocast(device_type='cuda'):
                    log_count, logits = modelo(lote)
            else:
                log_count, logits = modelo(lote)
            contagens.append(torch.expm1(log_count.float().clamp(min=0)).cpu().numpy())
            scores.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(contagens), np.concatenate(scores)


def perda_calibracao(modelo, x, y_log, y_bin, criterio_reg, criterio_cls, pos_weight):
    modelo.eval()
    total, quantidade = 0.0, 0
    with torch.inference_mode():
        for inicio in range(0, len(x), BATCH_SIZE):
            fim = inicio + BATCH_SIZE
            if use_cuda_amp:
                with torch.amp.autocast(device_type='cuda'):
                    pred_log, logits = modelo(x[inicio:fim])
                    perda = PESO_REGRESSAO * criterio_reg(pred_log, y_log[inicio:fim]) + criterio_cls(logits, y_bin[inicio:fim])
            else:
                pred_log, logits = modelo(x[inicio:fim])
                perda = PESO_REGRESSAO * criterio_reg(pred_log, y_log[inicio:fim]) + criterio_cls(logits, y_bin[inicio:fim])
            total += float(perda) * min(BATCH_SIZE, len(x) - inicio)
            quantidade += min(BATCH_SIZE, len(x) - inicio)
    return total / max(quantidade, 1)


resultados_nn = []
resultados_tabulares = []
for horizonte in CONFIG['treino']['horizontes']:
    inicio_tempo = pd.Timestamp(inicio_ano_epi(min(CONFIG['treino']['validacao_anos'])))
    fim_validacao = pd.Timestamp(inicio_ano_epi(max(CONFIG['treino']['validacao_anos']) + 1))
    inicio_calibracao = inicio_tempo - pd.Timedelta(weeks=int(CONFIG['treino']['calibracao_semanas']))
    alvo_data = f'data_alvo_h{horizonte}'
    elegivel = mascara_elegivel(dados, horizonte).to_numpy()
    datas_alvo = pd.to_datetime(dados[alvo_data])
    treino_base = elegivel & conhecidos.to_numpy() & (datas_alvo < inicio_calibracao).to_numpy()
    calibracao_base = elegivel & conhecidos.to_numpy() & (datas_alvo >= inicio_calibracao).to_numpy() & (datas_alvo < inicio_tempo).to_numpy()
    validacao_base = elegivel & conhecidos.to_numpy() & (datas_alvo >= inicio_tempo).to_numpy() & (datas_alvo < fim_validacao).to_numpy()
    indices_treino = np.flatnonzero(treino_base)
    indices_calibracao = np.flatnonzero(calibracao_base)
    indices_validacao = np.flatnonzero(validacao_base)
    if min(len(indices_treino), len(indices_calibracao), len(indices_validacao)) == 0:
        raise ValueError(f'Split temporal vazio em h{horizonte}: treino/calibração/validação.')

    print(f"\n--- Horizonte h{horizonte}: treino até {inicio_calibracao.date()}, calibração antes de {inicio_tempo.date()}, validação 2025 ---")
    preprocessador = None
    X_fit, y_fit_nn, df_fit_nn, preprocessador = preparar_tensores(
        dados, indices_treino, horizonte, features_input, indices_treino, SEQ_LEN, preprocessador
    )
    X_cal, y_cal_nn, df_cal_nn, preprocessador = preparar_tensores(
        dados, indices_calibracao, horizonte, features_input, indices_treino, SEQ_LEN, preprocessador
    )
    X_val, y_val_nn, df_val_nn, preprocessador = preparar_tensores(
        dados, indices_validacao, horizonte, features_input, indices_treino, SEQ_LEN, preprocessador
    )

    # Tensores completos são transferidos uma única vez por split/horizonte.
    X_fit_dev, X_cal_dev, X_val_dev = (tensor.to(device) for tensor in (X_fit, X_cal, X_val))
    y_fit_log = torch.log1p(y_fit_nn.clamp(min=0)).to(device)
    y_cal_log = torch.log1p(y_cal_nn.clamp(min=0)).to(device)
    y_fit_bin = (y_fit_nn >= CASOS_MINIMOS).float().to(device)
    y_cal_bin = (y_cal_nn >= CASOS_MINIMOS).float().to(device)
    positivos = float(y_fit_bin.sum().item())
    negativos = float(len(y_fit_bin) - positivos)
    if positivos < 1 or negativos < 1:
        raise ValueError(f'Classe sem exemplos positivos/negativos no treino de h{horizonte}.')
    pos_weight = torch.tensor([min(negativos / positivos, 50.0)], dtype=torch.float32, device=device)
    criterio_reg = nn.HuberLoss(delta=1.0)
    criterio_cls = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    tabular = Candidato(CONFIG, 'hurdle').fit(
        dados.iloc[indices_treino], dados.iloc[indices_treino][f'alvo_h{horizonte}'],
        schema['numericas'], schema['categoricas'], data_alvo=alvo_data,
    )
    tab_cal_prob = tabular.predict_ocorrencia(df_cal_nn)
    tab_val_prob = tabular.predict_ocorrencia(df_val_nn)
    tab_cal_count = tabular.predict(df_cal_nn)
    tab_val_count = tabular.predict(df_val_nn)
    tab_limiar, _ = escolher_limiar(y_cal_nn.numpy(), tab_cal_prob)
    resultados_nn.append({
        'horizonte': f'h{horizonte}', 'modelo': 'CatBoost Hurdle (reajuste temporal)',
        **medir_modelo(y_val_nn.numpy(), tab_val_count, tab_val_prob, df_val_nn, tab_limiar),
    })

    # Para a tabela de todos os candidatos, reutiliza o relatório tabular 2025.
    relatorio = pasta_treino / f'validacao_h{horizonte}.json'
    if relatorio.is_file():
        for item in json.loads(relatorio.read_text()):
            met = item.get('metricas', {})
            ocorrencia = met.get('ocorrencia')
            if item.get('grupo') != 'conhecidos' or not ocorrencia:
                continue
            nome = item['modelo']
            resultados_tabulares.append({
                'horizonte': f'h{horizonte}', 'modelo': nome,
                'PR-AUC': ocorrencia.get('pr_auc'), 'F1': ocorrencia.get('f1'),
                'Recall': ocorrencia.get('recall'), 'Precisão': ocorrencia.get('precision'),
                'MAE macro': met.get('macro_mae'), 'Limiar': np.nan,
                'Meta P/R >= 0.80': bool((ocorrencia.get('precision') or 0) >= PRECISAO_MINIMA and (ocorrencia.get('recall') or 0) >= RECALL_MINIMO),
            })

    baseline_specs = [
        ('Baseline zero', lambda ref: np.zeros(len(ref), dtype=float)),
        ('Baseline recente', lambda ref: ref.baseline_recente.to_numpy(dtype=float)),
        ('Baseline média 4s', lambda ref: ref.baseline_media4.to_numpy(dtype=float)),
        (f'Baseline sazonal h{horizonte}', lambda ref: ref[f'baseline_sazonal_h{horizonte}'].to_numpy(dtype=float)),
    ]
    for nome, get_counts in baseline_specs:
        cal_counts = np.maximum(get_counts(df_cal_nn), 0)
        val_counts = np.maximum(get_counts(df_val_nn), 0)
        limiar, _ = escolher_limiar(y_cal_nn.numpy(), cal_counts)
        resultados_nn.append({
            'horizonte': f'h{horizonte}', 'modelo': nome,
            **medir_modelo(y_val_nn.numpy(), val_counts, val_counts, df_val_nn, limiar),
        })

    contagens_cal, contagens_val = {}, {}
    scores_cal, scores_val = {}, {}
    y_fit_train = y_fit_log

    for posicao, (nome, arquitetura) in enumerate(ARQUITETURAS.items(), start=1):
        torch.manual_seed(CONFIG['semente'] + horizonte * 100 + posicao)
        argumentos = {'seq_len': SEQ_LEN} if nome == 'DengueResMLP' else {}
        modelo = arquitetura(X_fit.shape[2], **argumentos).to(device)
        amp_scaler = torch.amp.GradScaler('cuda', enabled=use_cuda_amp)
        otimizador = torch.optim.AdamW(
            modelo.parameters(), lr=float(NEURAL_CFG.get('taxa_aprendizado', 0.001)), weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(otimizador, T_max=EPOCHS)
        melhor_perda, sem_melhora, melhor_estado = float('inf'), 0, None
        tempo_modelo = time.perf_counter()

        for epoca in range(EPOCHS):
            modelo.train()
            ordem = torch.randperm(len(X_fit_dev), device=device)
            for inicio in range(0, len(ordem), BATCH_SIZE):
                indices_lote = ordem[inicio:inicio + BATCH_SIZE]
                otimizador.zero_grad(set_to_none=True)
                if use_cuda_amp:
                    with torch.amp.autocast(device_type='cuda'):
                        pred_log, logits = modelo(X_fit_dev[indices_lote])
                        perda_reg = criterio_reg(pred_log, y_fit_train[indices_lote])
                        perda_cls = criterio_cls(logits, y_fit_bin[indices_lote])
                        perda = PESO_REGRESSAO * perda_reg + perda_cls
                else:
                    pred_log, logits = modelo(X_fit_dev[indices_lote])
                    perda_reg = criterio_reg(pred_log, y_fit_train[indices_lote])
                    perda_cls = criterio_cls(logits, y_fit_bin[indices_lote])
                    perda = PESO_REGRESSAO * perda_reg + perda_cls
                if not torch.isfinite(perda):
                    raise FloatingPointError(f'Loss não finita em {nome}/h{horizonte}, época {epoca + 1}.')
                amp_scaler.scale(perda).backward()
                amp_scaler.unscale_(otimizador)
                nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
                amp_scaler.step(otimizador)
                amp_scaler.update()
            scheduler.step()
            perda_cal = perda_calibracao(
                modelo, X_cal_dev, y_cal_log, y_cal_bin, criterio_reg, criterio_cls, pos_weight
            )
            if perda_cal < melhor_perda - 1e-5:
                melhor_perda = perda_cal
                melhor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
                sem_melhora = 0
            else:
                sem_melhora += 1
                if sem_melhora >= PATIENCE:
                    break

        if melhor_estado is None:
            raise RuntimeError(f'Early stopping não produziu estado válido para {nome}/h{horizonte}.')
        modelo.load_state_dict(melhor_estado)
        cal_count, cal_score = inferir_lotes(modelo, X_cal_dev)
        val_count, val_score = inferir_lotes(modelo, X_val_dev)
        threshold, atingiu_cal = escolher_limiar(y_cal_nn.numpy(), cal_score)
        metricas_val = medir_modelo(y_val_nn.numpy(), val_count, val_score, df_val_nn, threshold)
        resultados_nn.append({'horizonte': f'h{horizonte}', 'modelo': nome, **metricas_val})
        scores_cal[nome], scores_val[nome] = cal_score, val_score
        contagens_cal[nome], contagens_val[nome] = cal_count, val_count
        predicoes_cal[nome], predicoes_val[nome] = cal_score, val_score
        print(
            f"⏳ [{posicao}/7] Treinando {nome}/h{horizonte}... ✔ {time.perf_counter()-tempo_modelo:.1f}s | "
            f"Recall: {metricas_val['Recall']:.3f} | Precisão: {metricas_val['Precisão']:.3f} | "
            f"PR-AUC: {metricas_val['PR-AUC']:.3f}"
        )
        del modelo, melhor_estado

    ranking = sorted(
        ARQUITETURAS.keys(),
        key=lambda nome: (
            average_precision_score((y_cal_nn.numpy() >= CASOS_MINIMOS).astype(int), scores_cal[nome])
            if np.any(y_cal_nn.numpy() >= CASOS_MINIMOS) else -1,
            -medir_modelo(y_cal_nn.numpy(), contagens_cal[nome], scores_cal[nome], df_cal_nn,
                          escolher_limiar(y_cal_nn.numpy(), scores_cal[nome])[0])['MAE macro'],
        ), reverse=True,
    )
    top4 = ranking[:4]
    prob_top4_cal = np.mean([scores_cal[name] for name in top4], axis=0)
    prob_top4_val = np.mean([scores_val[name] for name in top4], axis=0)
    count_top4_cal = np.median([contagens_cal[name] for name in top4], axis=0)
    count_top4_val = np.median([contagens_val[name] for name in top4], axis=0)
    top4_threshold, _ = escolher_limiar(y_cal_nn.numpy(), prob_top4_cal)
    resultados_nn.append({
        'horizonte': f'h{horizonte}', 'modelo': 'Ensemble Top-4 PyTorch',
        **medir_modelo(y_val_nn.numpy(), count_top4_val, prob_top4_val, df_val_nn, top4_threshold),
    })

    prob_meta_cal = (prob_top4_cal + tab_cal_prob) / 2
    prob_meta_val = (prob_top4_val + tab_val_prob) / 2
    count_meta_cal = np.median([count_top4_cal, tab_cal_count], axis=0)
    count_meta_val = np.median([count_top4_val, tab_val_count], axis=0)
    meta_threshold, _ = escolher_limiar(y_cal_nn.numpy(), prob_meta_cal)
    resultados_nn.append({
        'horizonte': f'h{horizonte}', 'modelo': 'Meta-Ensemble Híbrido',
        **medir_modelo(y_val_nn.numpy(), count_meta_val, prob_meta_val, df_val_nn, meta_threshold),
    })
    print(f"Top-4/h{horizonte}: {', '.join(top4)} | limiar híbrido calibrado em dados pré-2025: {meta_threshold:.4f}")

    del X_fit_dev, X_cal_dev, X_val_dev, y_fit_log, y_cal_log, y_fit_bin, y_cal_bin
    if device.type == 'cuda':
        torch.cuda.empty_cache()

# Comparison includes every tabular candidate report plus baselines, networks and ensembles.
comparacao = pd.DataFrame(resultados_tabulares + resultados_nn)
comparacao = comparacao.sort_values(
    ['horizonte', 'PR-AUC', 'F1', 'Recall', 'Precisão', 'MAE macro'],
    ascending=[True, False, False, False, False, True], na_position='last',
).reset_index(drop=True)
display(comparacao)


In [ ]:
import os

def preparar_caminhos_locais():
    """
    Configura o script para ler os arquivos de dados na pasta de execução local corrente
    """
    diretorio_corrente = os.getcwd()
    parquet_local = os.path.join(diretorio_corrente, 'dengue-analitico.parquet')

    print("--- Configuração de Ambiente Local ---")
    print(f"Pasta local identificada: {diretorio_corrente}")
    print(f"Caminho esperado para o dataset local: {parquet_local}")

    if os.path.exists(parquet_local):
        print("✔ Dataset local encontrado com sucesso!")
    else:
        print("⚠ Dataset não encontrado na pasta local corrente. Certifique-se de manter o arquivo 'dengue-analitico.parquet' no mesmo diretório ao executar localmente.")

preparar_caminhos_locais()